# Figure 6 and Supplementary Figure 8 analysis


In [ ]:
from pathlib import Path
import os

FIG6_PROJECT_ROOT = Path('/mnt/disk18t/lr_xcy/riku/kirc_205')
FIG6_CM2_ROOT = FIG6_PROJECT_ROOT.parent / 'codex_research/CM_analysis_2'
FIG6_SSGSEA_ROOT = FIG6_CM2_ROOT / 'cm_ssgsea'
FIG6_COX_ROOT = FIG6_SSGSEA_ROOT / 'cm_survival/Cox_analysis'
FIG6_KM_ROOT = FIG6_SSGSEA_ROOT / 'cm_survival/KM_analysis'
FIG6_THERAPY_COX_ROOT = FIG6_SSGSEA_ROOT / 'cm_survival/Immune_therapy_cox'
FIG6_THERAPY_KM_ROOT = FIG6_SSGSEA_ROOT / 'cm_survival/Immune_therapy_KM'
FIG6_REVISION_ROOT = FIG6_PROJECT_ROOT / 'review_revision_20260819'


def sync_fig6_bulk_score_tables():
    # Original downstream folders hold local copies of these four ssGSEA outputs.
    import shutil
    destinations = {
        FIG6_COX_ROOT: ['nes_diff.csv', 'nes_diff_cm.csv', 'nes_diff_javelin.csv', 'nes_diff_javelin_cm.csv'],
        FIG6_KM_ROOT: ['nes_diff.csv', 'nes_diff_cm.csv', 'nes_diff_javelin.csv', 'nes_diff_javelin_cm.csv'],
        FIG6_THERAPY_COX_ROOT: ['nes_diff_javelin.csv', 'nes_diff_javelin_cm.csv'],
        FIG6_THERAPY_KM_ROOT: ['nes_diff_javelin.csv', 'nes_diff_javelin_cm.csv'],
    }
    for directory, filenames in destinations.items():
        for filename in filenames:
            source = FIG6_SSGSEA_ROOT / filename
            if not source.is_file():
                raise FileNotFoundError(source)
    for directory, filenames in destinations.items():
        directory.mkdir(parents=True, exist_ok=True)
        for filename in filenames:
            shutil.copy2(FIG6_SSGSEA_ROOT / filename, directory / filename)


## 0. CM reference expansion and marker export


In [ ]:
# Source: cm_anno/map_cell_subtype_to_cm_expand_adata.ipynb
def rebuild_fig6_cm_expanded_reference():
    from collections import defaultdict
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import scanpy as sc

    ADATA_PATH = FIG6_CM2_ROOT / "cm_anno/adata_anno_cell_subtype_re.h5ad"
    MAPPING_CSV = FIG6_CM2_ROOT / "cm_anno/balanced_joint_cm_reference_node_sets_after_edge_threshold.csv"
    SUBTYPE_COL = "cell_subtype"

    # How to handle cell_subtype values absent from the CM-node CSV:
    #   "fill_subtype" -> keep those cells with CM set to their own subtype name
    #   "drop"         -> exclude those cells from the expanded output
    #   "error"        -> stop if any subtype is absent from the CSV
    UNMAPPED_MODE = "fill_subtype"

    OUTPUT_H5AD = FIG6_CM2_ROOT / "cm_anno/adata_anno_cell_subtype_re_cm_expanded.h5ad"
    OUTPUT_FRACTION_CSV = FIG6_CM2_ROOT / "cm_anno/cm_cell_subtype_fraction.csv"

    # The input file has .raw. Expanding .raw can make the output much larger.
    # Set False only if you explicitly need raw preserved in the expanded h5ad.
    DROP_RAW = False
    COMPRESSION = "gzip"

    mapping_df = pd.read_csv(MAPPING_CSV)
    required = {"CM", "node"}
    missing = required - set(mapping_df.columns)
    if missing:
        raise ValueError(f"Mapping CSV missing required columns: {sorted(missing)}")

    mapping_df = mapping_df.copy()
    if "rank" not in mapping_df.columns:
        if "reference_node_rank" in mapping_df.columns:
            mapping_df["rank"] = mapping_df["reference_node_rank"]
        else:
            mapping_df["rank"] = mapping_df.groupby("CM").cumcount() + 1
    if "weight" not in mapping_df.columns:
        mapping_df["weight"] = np.nan

    mapping_df["CM"] = mapping_df["CM"].astype(str)
    mapping_df["node"] = mapping_df["node"].astype(str)
    mapping_df["rank"] = mapping_df["rank"].astype(int)
    mapping_df["weight"] = mapping_df["weight"].astype(float)
    mapping_df = mapping_df.sort_values(["CM", "rank", "node"]).reset_index(drop=True)

    duplicated_pairs = mapping_df.duplicated(["CM", "node"], keep=False)
    if duplicated_pairs.any():
        dup = mapping_df.loc[duplicated_pairs, ["CM", "node"]].drop_duplicates()
        raise ValueError("Duplicate CM-node pairs in mapping CSV:\n" + dup.to_string(index=False))

    node_to_maps = defaultdict(list)
    for row in mapping_df.itertuples(index=False):
        node_to_maps[row.node].append({
            "CM": row.CM,
            "CM_node": row.node,
            "CM_weight": row.weight,
            "CM_rank": row.rank,
        })

    cm_order = mapping_df["CM"].drop_duplicates().tolist()

    print("CM order:", cm_order)
    print("mapping rows:", len(mapping_df))
    print("unique nodes:", mapping_df["node"].nunique())
    print("nodes mapping to multiple CMs:", sum(mapping_df.groupby("node")["CM"].nunique() > 1))

    mapping_df.head(20)

    adata = sc.read_h5ad(ADATA_PATH)
    if SUBTYPE_COL not in adata.obs.columns:
        raise KeyError(f"{SUBTYPE_COL!r} not found in adata.obs")

    subtype_counts = adata.obs[SUBTYPE_COL].astype(str).value_counts()
    mapped_nodes = set(mapping_df["node"].astype(str))
    unmapped_counts = subtype_counts[~subtype_counts.index.isin(mapped_nodes)]

    expanded_mapped_n = sum(int(n) * len(node_to_maps[subtype]) for subtype, n in subtype_counts.items() if subtype in node_to_maps)
    expanded_keep_unmapped_n = expanded_mapped_n + int(unmapped_counts.sum())

    print("adata shape:", adata.shape)
    print("cell_subtype count:", len(subtype_counts))
    print("mapped subtype count:", int(subtype_counts.index.isin(mapped_nodes).sum()))
    print("unmapped subtype count:", len(unmapped_counts))
    print("original cells:", adata.n_obs)
    print("expanded cells, mapped only:", expanded_mapped_n)
    print("expanded cells, fill subtype:", expanded_keep_unmapped_n)

    unmapped_counts.to_frame("n_cells")

    def build_expansion_index(obs, subtype_col, node_to_maps, unmapped_mode):
        subtypes = obs[subtype_col].astype(str).to_numpy()
        obs_names = obs.index.astype(str).to_numpy()

        row_indices = []
        cm_values = []
        cm_nodes = []
        cm_weights = []
        cm_ranks = []
        mapping_status = []
        missing_subtypes = defaultdict(int)

        for idx, subtype in enumerate(subtypes):
            maps = node_to_maps.get(subtype)
            if maps:
                for item in maps:
                    row_indices.append(idx)
                    cm_values.append(item["CM"])
                    cm_nodes.append(item["CM_node"])
                    cm_weights.append(item["CM_weight"])
                    cm_ranks.append(item["CM_rank"])
                    mapping_status.append("mapped")
            else:
                missing_subtypes[subtype] += 1
                if unmapped_mode == "fill_subtype":
                    row_indices.append(idx)
                    cm_values.append(subtype)
                    cm_nodes.append(subtype)
                    cm_weights.append(np.nan)
                    cm_ranks.append(np.nan)
                    mapping_status.append("filled_subtype")
                elif unmapped_mode == "drop":
                    continue
                elif unmapped_mode == "error":
                    continue
                else:
                    raise ValueError(f"Unknown unmapped_mode: {unmapped_mode}")

        if missing_subtypes and unmapped_mode == "error":
            missing_df = (
                pd.Series(missing_subtypes, name="n_cells")
                .sort_values(ascending=False)
                .rename_axis(subtype_col)
                .reset_index()
            )
            raise ValueError(
                "Some cell_subtype values are absent from the CM-node mapping CSV. "
                "Use UNMAPPED_MODE='fill_subtype' or UNMAPPED_MODE='drop' only if intended.\n"
                + missing_df.to_string(index=False)
            )

        row_indices = np.asarray(row_indices, dtype=np.int64)
        cm_values = np.asarray(cm_values, dtype=object)
        new_obs_names = np.asarray(
            [f"{obs_names[i]}__CM_{cm}" for i, cm in zip(row_indices, cm_values)],
            dtype=object,
        )

        return {
            "row_indices": row_indices,
            "new_obs_names": new_obs_names,
            "CM": cm_values,
            "CM_node": np.asarray(cm_nodes, dtype=object),
            "CM_weight": np.asarray(cm_weights, dtype=float),
            "CM_rank": np.asarray(cm_ranks, dtype=float),
            "CM_mapping_status": np.asarray(mapping_status, dtype=object),
            "missing_subtypes": dict(missing_subtypes),
        }


    def make_fraction_table(expanded_obs, subtype_col, mapping_df, cm_order, include_filled_subtypes):
        filled_subtypes = []
        if include_filled_subtypes and "filled_subtype" in set(expanded_obs["CM_mapping_status"].astype(str)):
            filled_subtypes = (
                expanded_obs.loc[expanded_obs["CM_mapping_status"].astype(str) == "filled_subtype", "CM"]
                .astype(str)
                .drop_duplicates()
                .sort_values()
                .tolist()
            )
        cm_categories = list(cm_order) + filled_subtypes

        count_df = (
            expanded_obs.groupby(["CM", subtype_col], observed=True)
            .size()
            .rename("n_cells")
            .reset_index()
        )

        own_nodes = mapping_df[["CM", "node", "weight", "rank"]].copy()
        own_nodes = own_nodes.rename(columns={"node": subtype_col, "weight": "CM_weight", "rank": "CM_rank"})
        if filled_subtypes:
            own_nodes = pd.concat([
                own_nodes,
                pd.DataFrame({"CM": filled_subtypes, subtype_col: filled_subtypes, "CM_weight": np.nan, "CM_rank": np.nan}),
            ], ignore_index=True)

        count_df = (
            own_nodes.merge(count_df, on=["CM", subtype_col], how="left")
            .assign(n_cells=lambda x: x["n_cells"].fillna(0).astype(int))
        )
        count_df["CM_total_cells"] = count_df.groupby("CM")["n_cells"].transform("sum")
        count_df["fraction_in_CM"] = np.where(
            count_df["CM_total_cells"] > 0,
            count_df["n_cells"] / count_df["CM_total_cells"],
            np.nan,
        )
        count_df["CM"] = pd.Categorical(count_df["CM"], categories=cm_categories, ordered=True)
        return count_df.sort_values(["CM", "CM_rank", subtype_col]).reset_index(drop=True)

    expansion = build_expansion_index(
        adata.obs,
        subtype_col=SUBTYPE_COL,
        node_to_maps=node_to_maps,
        unmapped_mode=UNMAPPED_MODE,
    )

    print("original cells:", adata.n_obs)
    print("expanded cells:", len(expansion["row_indices"]))
    print("unique output obs_names:", len(set(expansion["new_obs_names"])) == len(expansion["new_obs_names"]))

    expanded = adata[expansion["row_indices"], :].copy()
    expanded.obs = adata.obs.iloc[expansion["row_indices"]].copy()
    expanded.obs_names = expansion["new_obs_names"]
    expanded.obs["original_obs_name"] = adata.obs_names.astype(str).to_numpy()[expansion["row_indices"]]
    expanded.obs["CM"] = expansion["CM"]
    expanded.obs["CM_node"] = expansion["CM_node"]
    expanded.obs["CM_weight"] = expansion["CM_weight"]
    expanded.obs["CM_rank"] = expansion["CM_rank"]
    expanded.obs["CM_mapping_status"] = expansion["CM_mapping_status"]

    filled_subtypes = (
        expanded.obs.loc[expanded.obs["CM_mapping_status"].astype(str) == "filled_subtype", "CM"]
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    cm_categories = cm_order + filled_subtypes
    expanded.obs["CM"] = pd.Categorical(expanded.obs["CM"].astype(str), categories=cm_categories, ordered=True)
    expanded.obs["CM_mapping_status"] = pd.Categorical(
        expanded.obs["CM_mapping_status"].astype(str),
        categories=["mapped", "filled_subtype"],
    )

    # The expanded h5ad keeps unmapped subtypes as their own CM label, but the
    # CM-subtype proportion table should contain only nodes that belong to hCMs.
    fraction_df = make_fraction_table(
        expanded.obs,
        subtype_col=SUBTYPE_COL,
        mapping_df=mapping_df,
        cm_order=cm_order,
        include_filled_subtypes=False,
    )

    print("expanded shape:", expanded.shape)
    print("fraction table shape:", fraction_df.shape)
    fraction_df.head(20)

    if DROP_RAW:
        expanded.raw = None

    compression = None if str(COMPRESSION).lower() == "none" else COMPRESSION
    expanded.write_h5ad(OUTPUT_H5AD, compression=compression)
    fraction_df[["CM", SUBTYPE_COL, "fraction_in_CM"]].to_csv(OUTPUT_FRACTION_CSV, index=False)

    print("saved expanded adata:", OUTPUT_H5AD, expanded.shape)
    print("saved CM fraction CSV:", OUTPUT_FRACTION_CSV, fraction_df.shape)
    return OUTPUT_H5AD

rebuild_fig6_cm_expanded_reference()


In [ ]:
# Source: cm_degs/expanded_scanpy_ttest/run_expanded_scanpy_ttest.ipynb
def export_fig6_cm_degs():
    from pathlib import Path

    import pandas as pd
    import scanpy as sc

    BASE_DIR = FIG6_CM2_ROOT / 'cm_degs/expanded_scanpy_ttest'
    ADATA_PATH = FIG6_CM2_ROOT / 'cm_anno/adata_anno_cell_subtype_re_cm_expanded.h5ad'
    OUT_DIR = BASE_DIR / 'degs'
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    GROUP_COL = 'CM'
    CM_PREFIX = 'joint_'
    METHOD = 't-test'
    USE_RAW = True

    sc.settings.verbosity = 2
    print('adata:', ADATA_PATH)
    print('output:', OUT_DIR)


    adata = sc.read_h5ad(ADATA_PATH)
    if GROUP_COL not in adata.obs.columns:
        raise KeyError(f'{GROUP_COL!r} not found in adata.obs')

    mask = adata.obs[GROUP_COL].astype(str).str.startswith(CM_PREFIX)
    adata_cm = adata[mask].copy()
    if adata_cm.n_obs == 0:
        raise ValueError(f'No rows found with {GROUP_COL} starting with {CM_PREFIX!r}')
    adata_cm.obs[GROUP_COL] = adata_cm.obs[GROUP_COL].astype(str).astype('category')

    print('raw exists:', adata_cm.raw is not None)
    print('filtered shape:', adata_cm.shape)
    print(adata_cm.obs[GROUP_COL].value_counts().sort_index().to_string())


    n_genes = adata_cm.raw.n_vars if USE_RAW and adata_cm.raw is not None else adata_cm.n_vars

    sc.tl.rank_genes_groups(
        adata_cm,
        groupby=GROUP_COL,
        method=METHOD,
        reference='rest',
        use_raw=USE_RAW,
        n_genes=n_genes,
        corr_method='benjamini-hochberg',
    )


    result = adata_cm.uns['rank_genes_groups']
    groups = list(result['names'].dtype.names)
    field_map = {
        'names': 'gene',
        'scores': 'score',
        'logfoldchanges': 'logfoldchanges',
        'pvals': 'pvals',
        'pvals_adj': 'pvals_adj',
    }

    written = []
    for group in groups:
        data = {}
        for key, out_col in field_map.items():
            if key in result:
                data[out_col] = result[key][group]
        df = pd.DataFrame(data)
        out_path = OUT_DIR / f'{group}_ttest_deg.csv'
        df.to_csv(out_path, index=False)
        written.append(out_path)

    print('written files:', len(written))
    for path in written:
        print(path)

    return written

export_fig6_cm_degs()


## 1.1 TCGA epithelial ssGSEA


In [ ]:
os.chdir(FIG6_SSGSEA_ROOT)
import numpy as np
import pandas as pd
import gseapy as gp
from pathlib import Path

#Define a dictionary to store the results
top_100_cell_type_markers = {}
deg_dir = Path('degs_epi')
output_dir = Path('.')
output_dir.mkdir(exist_ok=True)

# Read the CSV files and store the top 100 genes in the dictionary
for deg_path in sorted(deg_dir.glob('*_degs_epi.csv')):
    subtype = deg_path.name.split('_deg')[0]
    top_100_cell_type_markers[subtype] = pd.read_csv(deg_path, index_col=0, nrows=100).index.tolist()

btm_100_cell_type_markers = {}

for deg_path in sorted(deg_dir.glob('*_degs_epi.csv')):
    subtype = deg_path.name.split('_deg')[0]
    btm_100_cell_type_markers[subtype] = pd.read_csv(deg_path, index_col=0).tail(100).index.tolist()

tcga_kirc_expr = pd.read_csv('gene_symbol_fpkm_star_kirc.csv',index_col=0)
tcga_kirc_expr

top_100_ss = gp.ssgsea(data=tcga_kirc_expr,
               gene_sets=top_100_cell_type_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

top_100_ss.res2d.sort_values('Name').head()

top_100_nes = top_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
top_100_nes_t = top_100_nes.T
top_100_nes_t.to_csv(output_dir / 'nes_top_100.csv')

btm_100_ss = gp.ssgsea(data=tcga_kirc_expr,
               gene_sets=btm_100_cell_type_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

btm_100_ss.res2d.sort_values('Name').head()

btm_100_nes = btm_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
btm_100_nes_t = btm_100_nes.T
btm_100_nes_t.to_csv(output_dir / 'nes_btm_100.csv')

nes_diff = top_100_nes_t-btm_100_nes_t
nes_diff

nes_diff.to_csv(output_dir / 'nes_diff.csv')


## 1.2 TCGA CM ssGSEA


In [ ]:
os.chdir(FIG6_SSGSEA_ROOT)
import numpy as np
import pandas as pd
import gseapy as gp
from pathlib import Path

#Define a dictionary to store the results
top_100_cm_markers = {}
deg_dir = Path('/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_degs/expanded_scanpy_ttest/degs')

# Read the CSV files and store the top 100 genes in the dictionary
for deg_path in sorted(deg_dir.glob('joint_*_ttest_deg.csv')):
    cm_name = deg_path.name.split('_ttest_deg')[0]
    top_100_cm_markers[cm_name] = pd.read_csv(deg_path, index_col=0, nrows=100).index.tolist()

btm_100_cm_markers = {}

for deg_path in sorted(deg_dir.glob('joint_*_ttest_deg.csv')):
    cm_name = deg_path.name.split('_ttest_deg')[0]
    btm_100_cm_markers[cm_name] = pd.read_csv(deg_path, index_col=0).tail(100).index.tolist()

btm_100_cm_markers

tcga_kirc_expr = pd.read_csv('gene_symbol_fpkm_star_kirc.csv',index_col=0)
tcga_kirc_expr

top_100_ss = gp.ssgsea(data=tcga_kirc_expr,
               gene_sets=top_100_cm_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

top_100_ss.res2d.sort_values('Name').head()

top_100_nes = top_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
top_100_nes_t = top_100_nes.T
top_100_nes_t.to_csv('nes_top_100_cm.csv')

btm_100_ss = gp.ssgsea(data=tcga_kirc_expr,
               gene_sets=btm_100_cm_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

btm_100_ss.res2d.sort_values('Name').head()

btm_100_nes = btm_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
btm_100_nes_t = btm_100_nes.T
btm_100_nes_t.to_csv('nes_btm_100_cm.csv')

nes_diff = top_100_nes_t-btm_100_nes_t
nes_diff

nes_diff.to_csv('nes_diff_cm.csv')


## 1.3 JAVELIN epithelial ssGSEA


In [ ]:
os.chdir(FIG6_SSGSEA_ROOT)
import numpy as np
import pandas as pd
import gseapy as gp
from pathlib import Path

#Define a dictionary to store the results
top_100_cell_type_markers = {}
deg_dir = Path('degs_epi')
output_dir = Path('.')
output_dir.mkdir(exist_ok=True)

# Read the CSV files and store the top 100 genes in the dictionary
for deg_path in sorted(deg_dir.glob('*_degs_epi.csv')):
    subtype = deg_path.name.split('_deg')[0]
    top_100_cell_type_markers[subtype] = pd.read_csv(deg_path, index_col=0, nrows=100).index.tolist()

btm_100_cell_type_markers = {}

for deg_path in sorted(deg_dir.glob('*_degs_epi.csv')):
    subtype = deg_path.name.split('_deg')[0]
    btm_100_cell_type_markers[subtype] = pd.read_csv(deg_path, index_col=0).tail(100).index.tolist()

btm_100_cell_type_markers

javelin_expr = pd.read_csv('javelin_gene_expression_log2tpm.csv',index_col=0)
javelin_expr = javelin_expr.groupby(javelin_expr.index).sum()

javelin_expr

top_100_ss = gp.ssgsea(data=javelin_expr,
               gene_sets=top_100_cell_type_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

top_100_ss.res2d.sort_values('Name').head()

top_100_nes = top_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
top_100_nes_t = top_100_nes.T
top_100_nes_t.to_csv(output_dir / 'nes_top_100_javelin.csv')

btm_100_ss = gp.ssgsea(data=javelin_expr,
               gene_sets=btm_100_cell_type_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

btm_100_ss.res2d.sort_values('Name').head()

btm_100_nes = btm_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
btm_100_nes_t = btm_100_nes.T
btm_100_nes_t.to_csv(output_dir / 'nes_btm_100_javelin.csv')

nes_diff = top_100_nes_t-btm_100_nes_t
nes_diff

nes_diff.to_csv(output_dir / 'nes_diff_javelin.csv')


## 1.4 JAVELIN CM ssGSEA


In [ ]:
os.chdir(FIG6_SSGSEA_ROOT)
import numpy as np
import pandas as pd
import gseapy as gp
from pathlib import Path

#Define a dictionary to store the results
top_100_cm_markers = {}
deg_dir = Path('/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_degs/expanded_scanpy_ttest/degs')

# Read the CSV files and store the top 100 genes in the dictionary
for deg_path in sorted(deg_dir.glob('joint_*_ttest_deg.csv')):
    cm_name = deg_path.name.split('_ttest_deg')[0]
    top_100_cm_markers[cm_name] = pd.read_csv(deg_path, index_col=0, nrows=100).index.tolist()

btm_100_cm_markers = {}

for deg_path in sorted(deg_dir.glob('joint_*_ttest_deg.csv')):
    cm_name = deg_path.name.split('_ttest_deg')[0]
    btm_100_cm_markers[cm_name] = pd.read_csv(deg_path, index_col=0).tail(100).index.tolist()

btm_100_cm_markers

javelin_expr = pd.read_csv('javelin_gene_expression_log2tpm.csv',index_col=0)
javelin_expr = javelin_expr.groupby(javelin_expr.index).sum()
javelin_expr

top_100_ss = gp.ssgsea(data=javelin_expr,
               gene_sets=top_100_cm_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

top_100_ss.res2d.sort_values('Name').head()

top_100_nes = top_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
top_100_nes_t = top_100_nes.T
top_100_nes_t.to_csv('nes_top_100_javelin_cm.csv')

btm_100_ss = gp.ssgsea(data=javelin_expr,
               gene_sets=btm_100_cm_markers,
               outdir=None,
               sample_norm_method='custom', # choose 'custom' will only use the raw value of `data`
               no_plot=True)

btm_100_ss.res2d.sort_values('Name').head()

btm_100_nes = btm_100_ss.res2d.pivot(index='Term', columns='Name', values='NES')
btm_100_nes_t = btm_100_nes.T
btm_100_nes_t.to_csv('nes_btm_100_javelin_cm.csv')

nes_diff = top_100_nes_t-btm_100_nes_t
nes_diff

nes_diff.to_csv('nes_diff_javelin_cm.csv')


## 2. KIRC Top10/Bottom10 Cox analysis and JAVELIN alignment


In [ ]:
sync_fig6_bulk_score_tables()
os.chdir(FIG6_COX_ROOT)
# -*- coding: utf-8 -*-
"""
Integrated workflow: select KIRC Top10 and Bottom10 by descending HR, then align JAVELIN plots
- Trivariate Cox model: EPI_z, CM_z, and INT=EPI_z x CM_z
- Retain HR, confidence interval, and p value for the INT term
- JAVELIN: convert PFS_P_CNSR to PFS_P_event, with 0 mapped to event=1
- Save outputs under cox_top10_bottom10
"""

import os
import re
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=ConvergenceWarning)
plt.ioff()

# Common configuration
LABEL_SEP = ' × '
EPS = 1e-12
BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / 'cox_top10_bottom10'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Data paths
KIRC = {
    'name': 'KIRC',
    'epi_path': BASE_DIR / 'nes_diff.csv',
    'cm_path': BASE_DIR / 'nes_diff_cm.csv',
    'surv_path': BASE_DIR / 'TCGA-KIRC.survival.csv',
    'surv_index_col': 'sample',
    'time_col': 'OS.time',
    'event_col': 'OS',
    'out_csv': OUT_DIR / 'KIRC_cox_INT_results_linear_top10_bottom10.csv',
    'out_pdf': OUT_DIR / 'KIRC_forest_INT_top10_bottom10_linear_tight.pdf',
}

JAVELIN = {
    'name': 'JAVELIN',
    'epi_path': BASE_DIR / 'nes_diff_javelin.csv',
    'cm_path': BASE_DIR / 'nes_diff_javelin_cm.csv',
    'surv_path': BASE_DIR / 'javelin_clinical_adata.csv',
    'surv_index_col': 'ID',
    'time_col': 'PFS_P',
    'event_col': 'PFS_P_event',
    'convert_event_from_cnsr_col': 'PFS_P_CNSR',
    'out_csv': OUT_DIR / 'JAVELIN_cox_INT_results_linear_top10_bottom10.csv',
    'out_pdf': OUT_DIR / 'JAVELIN_forest_INT_aligned_to_KIRC_top10_bottom10_linear_tight.pdf',
}


# Utilities
def clean_index_values(index: pd.Index) -> pd.Index:
    cleaned = pd.Index(index).astype(str).str.replace('.', '-', regex=False).str.rstrip('A')
    return pd.Index(cleaned)


def clean_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = clean_index_values(df.index)
    df = df[~df.index.duplicated(keep='first')]
    return df


def cm_key(name: str):
    m = re.fullmatch(r'CM(\d+)', name)
    if m:
        return (int(m.group(1)), name)
    m = re.match(r'JOINT_(\d+)', name)
    if m:
        return (int(m.group(1)), name)
    return (10**9, name)


def make_label(epi_name: str, cm_name: str) -> str:
    return f'{epi_name}{LABEL_SEP}{cm_name}'


def load_and_align(epi_path, cm_path, surv_path, time_col, event_col,
                   surv_index_col=0, convert_event_from_cnsr_col=None, dataset_name='dataset'):
    """Read, align, and return (epi_z, cm_z, surv)。"""
    epi = pd.read_csv(epi_path, index_col=0)
    cm = pd.read_csv(cm_path, index_col=0)
    surv = pd.read_csv(surv_path, index_col=surv_index_col)
    surv = surv[surv.index.notna()].copy()

    if convert_event_from_cnsr_col:
        cnsr = pd.to_numeric(surv[convert_event_from_cnsr_col], errors='coerce')
        surv[event_col] = np.where(cnsr.isna(), np.nan, np.where(cnsr.eq(0), 1, 0))

    surv[time_col] = pd.to_numeric(surv[time_col], errors='coerce')
    surv[event_col] = pd.to_numeric(surv[event_col], errors='coerce')
    surv = surv[surv[time_col].notnull() & surv[event_col].notnull()]

    epi = clean_index(epi)
    cm = clean_index(cm)
    surv = clean_index(surv)

    epi.columns = epi.columns.str.upper()
    cm.columns = cm.columns.str.upper()

    epi_cols = [c for c in epi.columns if c.startswith('EPI_')]
    cm_cols = [c for c in cm.columns if re.fullmatch(r'CM\d+', c) or c.startswith('JOINT_')]
    cm_cols = sorted(cm_cols, key=cm_key)

    if not epi_cols:
        raise ValueError(f'{dataset_name}: no EPI_ columns found')
    if not cm_cols:
        raise ValueError(f'{dataset_name}: no CM or JOINT_ columns found')

    common = surv.index.intersection(epi.index).intersection(cm.index)
    if common.empty:
        raise ValueError(f'{dataset_name}: survival / EPI / CM have no shared samples')

    surv = surv.loc[common].sort_index()
    epi = epi.loc[common, epi_cols].sort_index()
    cm = cm.loc[common, cm_cols].sort_index()

    sc_epi = StandardScaler()
    sc_cm = StandardScaler()
    epi_z = pd.DataFrame(sc_epi.fit_transform(epi), index=epi.index, columns=epi.columns)
    cm_z = pd.DataFrame(sc_cm.fit_transform(cm), index=cm.index, columns=cm.columns)

    event_n = int(surv[event_col].sum())
    print(f'{dataset_name}: aligned n={len(common)}, events={event_n}, EPI={len(epi_cols)}, CM={len(cm_cols)}')
    return epi_z, cm_z, surv


def fit_trivariate_INT(epi_z, cm_z, surv, time_col, event_col):
    """Fit trivariate Cox models for every pair and return only INT statistics."""
    rows = []
    cm_cols_sorted = sorted(cm_z.columns, key=cm_key)
    epi_cols_sorted = sorted(epi_z.columns)

    for e_col, c_col in product(epi_cols_sorted, cm_cols_sorted):
        df_pair = pd.DataFrame(index=surv.index)
        df_pair['EPI'] = epi_z[e_col]
        df_pair['CM'] = cm_z[c_col]
        df_pair['INT'] = df_pair['EPI'] * df_pair['CM']
        df_pair = pd.concat([surv[[time_col, event_col]], df_pair], axis=1).dropna()

        if any(df_pair[v].nunique() <= 1 for v in ['EPI', 'CM', 'INT']) or len(df_pair) < 5:
            continue

        try:
            cph = CoxPHFitter()
            cph.fit(df_pair, duration_col=time_col, event_col=event_col)
            s = cph.summary.loc['INT']
            coef = float(s['coef'])
            rows.append({
                'Epi': e_col,
                'CM': c_col,
                'coef': coef,
                'HR': float(np.exp(coef)),
                'HR_lower': float(np.exp(s['coef lower 95%'])),
                'HR_upper': float(np.exp(s['coef upper 95%'])),
                'p': float(s['p']),
                'n': int(df_pair.shape[0]),
            })
        except Exception as exc:
            print(f'Skip {e_col} × {c_col}: {type(exc).__name__}')
            continue

    out_df = pd.DataFrame(rows)
    if not out_df.empty:
        out_df = out_df.sort_values('HR', ascending=False).reset_index(drop=True)
        out_df['label'] = [make_label(e, c) for e, c in zip(out_df['Epi'], out_df['CM'])]
    return out_df


def _symmetric_linear_xlim_around_one(hr_lower, hr_upper, pad_ratio=0.05, min_span=0.1):
    low = float(np.nanmin(hr_lower))
    high = float(np.nanmax(hr_upper))
    d = max(0.0, 1.0 - low, high - 1.0)
    d *= (1.0 + float(pad_ratio))
    d = max(d, float(min_span) / 2.0)
    return max(1.0 - d, EPS), 1.0 + d


def plot_cox_forest(df_plot, out_pdf, title, top_n=None):
    if df_plot is None or df_plot.empty:
        print('No data to plot.')
        return

    data = df_plot.copy()
    if top_n is not None and top_n > 0:
        data = data.head(top_n).copy()

    xmin, xmax = _symmetric_linear_xlim_around_one(
        data['HR_lower'].values,
        data['HR_upper'].values,
        pad_ratio=0.05,
        min_span=0.1,
    )

    n = len(data)
    y = np.arange(n)
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * n)))
    ax.errorbar(
        data['HR'].values,
        y,
        xerr=[data['HR'].values - data['HR_lower'].values,
              data['HR_upper'].values - data['HR'].values],
        fmt='o', color='black', ecolor='gray', capsize=3,
    )
    ax.axvline(1, color='red', linestyle='--', lw=1)
    ax.set_yticks(y, labels=data['label'])
    ax.set_xlabel('Hazard Ratio (HR)')
    ax.set_title(title, pad=5)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(n - 0.5, -0.5)
    fig.subplots_adjust(top=0.96, bottom=0.08, left=0.33, right=0.98)
    plt.tight_layout(pad=0.2)
    fig.savefig(out_pdf, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Forest plot saved -> {out_pdf.resolve()}')


def run_dataset(config):
    epi_z, cm_z, surv = load_and_align(
        config['epi_path'], config['cm_path'], config['surv_path'],
        config['time_col'], config['event_col'],
        surv_index_col=config.get('surv_index_col', 0),
        convert_event_from_cnsr_col=config.get('convert_event_from_cnsr_col'),
        dataset_name=config['name'],
    )
    df = fit_trivariate_INT(epi_z, cm_z, surv, config['time_col'], config['event_col'])
    if df is None or df.empty:
        raise SystemExit(f"{config['name']} returned no results; stopping.")
    return df


# Main workflow
kirc_df_all = run_dataset(KIRC)
if len(kirc_df_all) > 20:
    kirc_df = pd.concat([kirc_df_all.head(10), kirc_df_all.tail(10)], axis=0).reset_index(drop=True)
else:
    kirc_df = kirc_df_all.copy()

kirc_df.to_csv(KIRC['out_csv'], index=False)
print(f"KIRC Cox INT Top10 and Bottom10 results -> {KIRC['out_csv'].resolve()}; total {len(kirc_df)} rows")
plot_cox_forest(
    kirc_df,
    KIRC['out_pdf'],
    title='KIRC — Cox: EPI_z, CM_z, INT=EPI_z×CM_z (Top10 & Bottom10 interactions)',
    top_n=None,
)

label_order = kirc_df['label'].tolist()

for config in [JAVELIN]:
    df_all = run_dataset(config)
    aligned = df_all[df_all['label'].isin(label_order)].copy()
    if aligned.empty:
        print(f"{config['name']} has no combinations overlapping the KIRC Top10 and Bottom10; skip plotting.")
        continue

    aligned['label'] = pd.Categorical(aligned['label'], categories=label_order, ordered=True)
    aligned = aligned.sort_values('label')
    aligned.to_csv(config['out_csv'], index=False)
    print(f"{config['name']} Cox INT results aligned to KIRC Top10 and Bottom10 -> {config['out_csv'].resolve()}; total {len(aligned)} rows")
    print(f"{config['name']} combinations overlapping KIRC Top10 and Bottom10: {len(aligned)} / reference {len(label_order)}")

    plot_cox_forest(
        aligned,
        config['out_pdf'],
        title=f"{config['name']} — Cox: EPI_z, CM_z, INT=EPI_z×CM_z (Aligned to KIRC Top10 & Bottom10)",
        top_n=None,
    )


## 3. JAVELIN zscore-then-group CM forest plots


In [ ]:
# Source: Immune_therapy_cox/run_zscore_then_group_cox.py
def prepare_fig6_javelin_cm_cox(write_outputs=True):
    import numpy as np
    import pandas as pd
    from lifelines import CoxPHFitter
    ROOT = FIG6_THERAPY_COX_ROOT
    def norm_label(series: pd.Series) -> pd.Series:
        return series.astype(str).str.strip().str.casefold()

    def zscore(df: pd.DataFrame) -> pd.DataFrame:
        mean = df.mean(axis=0)
        std = df.std(axis=0, ddof=0).replace(0, 1.0)
        return (df - mean) / std

    def fit_univariate(var_df: pd.DataFrame, surv: pd.DataFrame, time_col: str, event_col: str) -> pd.DataFrame:
        rows = []
        for col in var_df.columns:
            df = pd.concat([surv[[time_col, event_col]], var_df[[col]]], axis=1).dropna()
            if df.shape[0] < 5 or df[col].nunique() <= 1:
                continue
            try:
                cph = CoxPHFitter()
                cph.fit(df, duration_col=time_col, event_col=event_col)
                s = cph.summary.loc[col]
                rows.append(
                    {
                        "feature": col,
                        "HR": float(np.exp(s["coef"])),
                        "HR_lower": float(np.exp(s["coef lower 95%"])),
                        "HR_upper": float(np.exp(s["coef upper 95%"])),
                        "p": float(s["p"]),
                        "n": int(df.shape[0]),
                    }
                )
            except Exception as exc:
                rows.append({"feature": col, "fit_error": str(exc), "n": int(df.shape[0])})
        out = pd.DataFrame(rows)
        if out.empty or "HR" not in out:
            return out
        return out.sort_values("HR", ascending=False, na_position="last").reset_index(drop=True)

    def add_univariate_metadata(res: pd.DataFrame, cohort: str, group: str, endpoint: str, model: str, time_col: str, event_rule: str) -> pd.DataFrame:
        res = res.copy()
        meta = {
            "cohort": cohort,
            "group": group,
            "endpoint": endpoint,
            "model": model,
            "time_col": time_col,
            "event_col": "event",
            "event_rule": event_rule,
            "zscore_rule": "cohort-level z-score before treatment-arm split",
        }
        for key, value in meta.items():
            res[key] = value
        meta_cols = list(meta)
        return res[meta_cols + [c for c in res.columns if c not in meta_cols]]

    def load_javelin():
        epi = pd.read_csv(ROOT / "nes_diff_javelin.csv", index_col=0)
        cm = pd.read_csv(ROOT / "nes_diff_javelin_cm.csv", index_col=0)
        cm.columns = cm.columns.str.upper()
        surv = pd.read_csv(ROOT / "javelin_clinical_adata.csv", index_col=0)
        surv = surv[surv["PFS_P"].notnull() & surv["PFS_P_CNSR"].notnull()].copy()
        surv["event"] = (surv["PFS_P_CNSR"] == 0).astype(int)
        surv = surv.join(epi, how="inner").join(cm, how="inner")
        epi_z = zscore(surv[epi.columns])
        cm_z = zscore(surv[cm.columns])
        groups = {
            "JAVELIN_AVEAX_PFS": ("Avelumab+Axitinib", surv[norm_label(surv["TRT01P"]) == "avelumab+axitinib"], "PFS_P", "event"),
            "JAVELIN_SUNITINIB_PFS": ("Sunitinib", surv[norm_label(surv["TRT01P"]) == "sunitinib"], "PFS_P", "event"),
        }
        return epi, cm, surv, epi_z, cm_z, groups

    _, _, _, _, cm_z, groups = load_javelin()
    output_dir = ROOT / 'zscore_then_group/univariate_cox/cm_z'
    if write_outputs:
        output_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for name, (group, survival, time_col, event_col) in groups.items():
        result = fit_univariate(cm_z.loc[survival.index], survival, time_col, event_col)
        result = add_univariate_metadata(result, 'JAVELIN', group, 'PFS', 'CM_z', time_col, 'JAVELIN reversed: event = 1 if PFS_P_CNSR == 0')
        if 'fit_error' in result and result['fit_error'].notna().any():
            raise RuntimeError(f'{name}: a CM-only Cox model failed')
        if write_outputs:
            result.to_csv(output_dir / f'{name}.csv', index=False)
        results[name] = result
    return results


fig6_javelin_cm_cox = prepare_fig6_javelin_cm_cox()


In [ ]:
os.chdir(FIG6_THERAPY_COX_ROOT)
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
matplotlib.rcParams.update({
    "pdf.fonttype": 42,
    "font.family": "DejaVu Sans",
    "font.size": 8,
})
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


CM_DIR = Path.cwd() / "zscore_then_group" / "univariate_cox" / "cm_z"
AVEAX_CSV = CM_DIR / "JAVELIN_AVEAX_PFS.csv"
SUNITINIB_CSV = CM_DIR / "JAVELIN_SUNITINIB_PFS.csv"
AVEAX_PDF = CM_DIR / "JAVELIN_AVEAX_PFS.pdf"
SUNITINIB_PDF = (
    CM_DIR
    / "javelin_sunitinib_ordered_by_aveaxi_hr"
    / "JAVELIN_SUNITINIB_PFS_CM_z_ordered_by_AVEAX_HR.pdf"
)


def javelin_cm_symmetric_xlim(hr_lower: pd.Series, hr_upper: pd.Series, pad: float = 0.05, min_span: float = 0.1):
    distance = max(1 - float(hr_lower.min()), float(hr_upper.max()) - 1)
    distance = max(distance * (1 + pad), min_span / 2)
    return max(1 - distance, 1e-12), 1 + distance


def plot_javelin_cm_forest(data: pd.DataFrame, title: str, output_pdf: Path) -> None:
    if data.empty:
        raise ValueError(f"No rows available for {title}")
    plot_data = data.copy()
    plot_data["plot_feature"] = plot_data["feature"].astype(str).str.lower()
    y_positions = np.arange(len(plot_data))
    xmin, xmax = javelin_cm_symmetric_xlim(plot_data["HR_lower"], plot_data["HR_upper"])

    figure, axis = plt.subplots(figsize=(8, max(4, 0.35 * len(plot_data))))
    axis.errorbar(
        plot_data["HR"],
        y_positions,
        xerr=[
            plot_data["HR"] - plot_data["HR_lower"],
            plot_data["HR_upper"] - plot_data["HR"],
        ],
        fmt="o",
        color="black",
        ecolor="gray",
        capsize=3,
    )
    axis.axvline(1, color="red", linestyle="--")
    axis.set_yticks(y_positions, plot_data["plot_feature"])
    axis.set_xlim(xmin, xmax)
    axis.set_ylim(len(plot_data) - 0.5, -0.5)
    axis.set_xlabel("Hazard Ratio (HR)")
    axis.set_title(title)
    figure.subplots_adjust(left=0.38, right=0.98, top=0.95, bottom=0.08)
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_pdf, dpi=300, bbox_inches="tight")
    plt.close(figure)


aveax = pd.read_csv(AVEAX_CSV)
plot_javelin_cm_forest(aveax, "javelin_aveax_pfs - CM_z", AVEAX_PDF)

sunitinib = pd.read_csv(SUNITINIB_CSV)
aveax_order = aveax.sort_values("HR", ascending=False)["feature"].astype(str).tolist()
sunitinib["_order"] = pd.Categorical(
    sunitinib["feature"].astype(str),
    categories=aveax_order,
    ordered=True,
)
sunitinib = sunitinib.sort_values("_order").drop(columns="_order").reset_index(drop=True)
plot_javelin_cm_forest(
    sunitinib,
    "javelin_sunitinib_pfs ordered by javelin_aveax_pfs HR - CM_z",
    SUNITINIB_PDF,
)


## 4. TCGA-KIRC overall-survival KM analysis


In [ ]:
os.chdir(FIG6_KM_ROOT)
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["text.usetex"] = False
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from scipy.stats import chi2

warnings.filterwarnings("ignore", category=UserWarning)

PALETTE = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]
SIGNIFICANCE_THRESHOLD = 0.05


def clean_filename(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value.strip("_")


def format_pvalue(p_value):
    if not np.isfinite(p_value):
        return "NA"
    if p_value < 1e-4:
        return f"{p_value:.2e}"
    return f"{p_value:.4f}"


def set_index_string(df):
    df = df.copy()
    df.index = df.index.astype(str)
    return df


def read_scores(epi_file, cm_file, sample_style):
    epi = set_index_string(pd.read_csv(epi_file, index_col=0))
    cm = set_index_string(pd.read_csv(cm_file, index_col=0))

    if sample_style == "tcga_dot_to_dash":
        epi.index = epi.index.str.replace(".", "-", regex=False)
        cm.index = cm.index.str.replace(".", "-", regex=False)

    epi = epi.groupby(epi.index).mean()
    cm = cm.groupby(cm.index).mean()
    return epi, cm


def read_survival(config):
    if config["cohort"] == "tcga":
        survival = set_index_string(pd.read_csv(config["survival_file"], index_col=0))
        survival = survival[[config["time_col"], config["event_col"]]].copy()
        survival[config["event_col"]] = pd.to_numeric(survival[config["event_col"]], errors="coerce")
    elif config["cohort"] == "javelin":
        survival = set_index_string(pd.read_csv(config["survival_file"], index_col=0))
        survival = survival[[config["time_col"], config["raw_event_col"]]].copy()
        raw_event = pd.to_numeric(survival[config["raw_event_col"]], errors="coerce")
        # JAVELIN is the only cohort needing reversal: PFS_P_CNSR == 0 means event.
        survival[config["event_col"]] = np.where(raw_event == 0, 1, np.where(raw_event.notna(), 0, np.nan))
        survival = survival[[config["time_col"], config["event_col"]]].copy()
    else:
        raise ValueError(f"Unknown cohort: {config['cohort']}")

    survival[config["time_col"]] = pd.to_numeric(survival[config["time_col"]], errors="coerce")
    survival[config["event_col"]] = pd.to_numeric(survival[config["event_col"]], errors="coerce")
    survival = survival.dropna(subset=[config["time_col"], config["event_col"]])
    survival = survival[survival[config["time_col"]] > 0]
    survival[config["event_col"]] = survival[config["event_col"]].astype(int)
    survival = survival.groupby(survival.index).first()
    return survival


def km_confidence_interval(survival, greenwood):
    if survival <= 0:
        return 0.0, 0.0
    if not np.isfinite(greenwood):
        return 0.0, 1.0
    se_log = np.sqrt(max(greenwood, 0.0))
    z = 1.959963984540054
    lower = survival * np.exp(-z * se_log)
    upper = survival * np.exp(z * se_log)
    return float(max(0.0, lower)), float(min(1.0, upper))


def km_estimate(times, events):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    if len(times) == 0:
        return np.array([0.0]), np.array([1.0]), np.array([1.0]), np.array([1.0]), np.array([]), np.array([])

    event_times = np.sort(np.unique(times[events == 1]))
    x = [0.0]
    y = [1.0]
    lower = [1.0]
    upper = [1.0]
    survival = 1.0
    greenwood = 0.0
    survival_after_event = {}

    for t in event_times:
        n_risk = np.sum(times >= t)
        n_events = np.sum((times == t) & (events == 1))
        if n_risk <= 0:
            continue
        survival *= (1.0 - n_events / n_risk)
        if n_risk > n_events:
            greenwood += n_events / (n_risk * (n_risk - n_events))
        else:
            greenwood = np.inf
        ci_lower, ci_upper = km_confidence_interval(survival, greenwood)
        x.append(float(t))
        y.append(float(survival))
        lower.append(ci_lower)
        upper.append(ci_upper)
        survival_after_event[float(t)] = float(survival)

    max_time = float(np.max(times))
    if x[-1] < max_time:
        x.append(max_time)
        y.append(float(survival))
        lower.append(float(lower[-1]))
        upper.append(float(upper[-1]))

    censor_times = np.sort(times[events == 0])
    censor_y = []
    for t in censor_times:
        s = 1.0
        for et in event_times[event_times <= t]:
            s = survival_after_event.get(float(et), s)
        censor_y.append(s)

    return np.asarray(x), np.asarray(y), np.asarray(lower), np.asarray(upper), censor_times, np.asarray(censor_y)


def multigroup_logrank(df, time_col, event_col, group_col, groups):
    data = df[df[group_col].isin(groups)].copy()
    if data[group_col].nunique() < 2:
        return np.nan

    times = data[time_col].to_numpy(dtype=float)
    events = data[event_col].to_numpy(dtype=int)
    group_values = data[group_col].to_numpy()
    event_times = np.sort(np.unique(times[events == 1]))
    k = len(groups)
    observed = np.zeros(k)
    expected = np.zeros(k)
    variance = np.zeros((k, k))

    for t in event_times:
        at_risk = times >= t
        at_event = (times == t) & (events == 1)
        n_total = int(at_risk.sum())
        d_total = int(at_event.sum())
        if n_total <= 1 or d_total == 0:
            continue

        n_by_group = np.array([np.sum(at_risk & (group_values == g)) for g in groups], dtype=float)
        d_by_group = np.array([np.sum(at_event & (group_values == g)) for g in groups], dtype=float)
        p = n_by_group / n_total
        observed += d_by_group
        expected += d_total * p
        factor = d_total * (n_total - d_total) / (n_total - 1)
        variance += factor * (np.diag(p) - np.outer(p, p))

    diff = observed - expected
    if np.allclose(variance, 0):
        return np.nan
    stat = float(diff @ np.linalg.pinv(variance) @ diff)
    dfree = max(k - 1, 1)
    return float(chi2.sf(stat, dfree))


def pairwise_logrank_results(df, time_col, event_col, group_col, groups, threshold=SIGNIFICANCE_THRESHOLD):
    all_results = []
    significant_results = []
    for i, group1 in enumerate(groups):
        for group2 in groups[i + 1:]:
            pair_df = df[df[group_col].isin([group1, group2])]
            if pair_df[group_col].nunique() < 2:
                continue
            p_value = multigroup_logrank(pair_df, time_col, event_col, group_col, [group1, group2])
            result = {"Group1": group1, "Group2": group2, "p_value": p_value}
            all_results.append(result)
            if np.isfinite(p_value) and p_value < threshold:
                significant_results.append(result)
    all_results.sort(key=lambda item: item["p_value"] if np.isfinite(item["p_value"]) else np.inf)
    significant_results.sort(key=lambda item: item["p_value"] if np.isfinite(item["p_value"]) else np.inf)
    return all_results, significant_results


def format_pairwise_text(significant_results, label_map=None, threshold=SIGNIFICANCE_THRESHOLD):
    if label_map is None:
        label_map = {}
    if not significant_results:
        return f"No significant pairwise log-rank tests (p < {threshold})."
    lines = [f"Significant pairwise log-rank tests (p < {threshold}):"]
    for result in significant_results:
        group1 = label_map.get(result['Group1'], result['Group1'])
        group2 = label_map.get(result['Group2'], result['Group2'])
        lines.append(f"{group1} vs {group2}: p = {format_pvalue(result['p_value'])}")
    return "\n".join(lines)


def serialize_pairwise_results(results, label_map=None):
    if label_map is None:
        label_map = {}
    if not results:
        return ""
    return "; ".join(
        f"{label_map.get(item['Group1'], item['Group1'])} vs {label_map.get(item['Group2'], item['Group2'])}: p={format_pvalue(item['p_value'])}"
        for item in results
    )


def get_time_ticks(max_time, break_by):
    upper = break_by * np.ceil(max_time / break_by)
    ticks = np.arange(0, upper + 0.5 * break_by, break_by)
    return ticks, float(upper)


def draw_censor_path(ax, censor_x, censor_y, color, x_upper):
    if len(censor_x) == 0:
        return
    dx = max(x_upper * 0.0035, 0.03)
    dy = 0.012
    verts = []
    codes = []
    for x_val, y_val in zip(censor_x, censor_y):
        verts.extend([(x_val - dx, y_val), (x_val + dx, y_val), (x_val, y_val - dy), (x_val, y_val + dy)])
        codes.extend([Path.MOVETO, Path.LINETO, Path.MOVETO, Path.LINETO])
    patch = PathPatch(
        Path(verts, codes),
        facecolor="none",
        edgecolor=color,
        lw=0.9,
        capstyle="butt",
        joinstyle="miter",
        zorder=3,
        clip_on=True,
    )
    ax.add_patch(patch)


def draw_pairwise_results(pair_ax, significant_results, label_map, color_map, threshold=SIGNIFICANCE_THRESHOLD):
    pair_ax.axis("off")
    y = 0.98
    line_step = 0.24
    pair_ax.text(
        0.0,
        y,
        f"Significant pairwise log-rank tests (p < {threshold}):",
        ha="left",
        va="top",
        fontsize=7.2,
        color="black",
        transform=pair_ax.transAxes,
    )
    y -= line_step
    if not significant_results:
        pair_ax.text(
            0.0,
            y,
            "No significant pairwise log-rank tests.",
            ha="left",
            va="top",
            fontsize=7.2,
            color="black",
            transform=pair_ax.transAxes,
        )
        return

    x_group1 = 0.000
    x_vs = 0.072
    x_group2 = 0.112
    x_p = 0.186
    for result in significant_results:
        group1 = label_map.get(result["Group1"], result["Group1"])
        group2 = label_map.get(result["Group2"], result["Group2"])
        pair_ax.text(x_group1, y, group1, ha="left", va="top", fontsize=7.2, color=color_map.get(group1, "black"), transform=pair_ax.transAxes)
        pair_ax.text(x_vs, y, "vs", ha="left", va="top", fontsize=7.2, color="black", transform=pair_ax.transAxes)
        pair_ax.text(x_group2, y, group2, ha="left", va="top", fontsize=7.2, color=color_map.get(group2, "black"), transform=pair_ax.transAxes)
        pair_ax.text(x_p, y, f"p = {format_pvalue(result['p_value'])}", ha="left", va="top", fontsize=7.2, color="black", transform=pair_ax.transAxes)
        y -= line_step


def draw_risk_table(risk_ax, current_df, time_col, group_col, plotted_groups, plotted_colors, plotted_labels, ticks, x_label, x_left, x_upper):
    n_rows = len(plotted_groups)
    risk_ax.set_xlim(x_left, x_upper)
    risk_ax.set_xticks(ticks)
    risk_ax.set_ylim(-0.75, n_rows + 1.2)
    risk_ax.set_yticks([n_rows - row_idx - 0.5 for row_idx in range(n_rows)])
    risk_ax.set_yticklabels(plotted_labels, fontsize=7.0)
    for tick_label, color in zip(risk_ax.get_yticklabels(), plotted_colors):
        tick_label.set_color(color)

    # Match ggsurvplot-like risk table: aligned time axis, no grid, only left and bottom borders.
    risk_ax.spines["left"].set_visible(True)
    risk_ax.spines["bottom"].set_visible(True)
    risk_ax.spines["left"].set_linewidth(0.8)
    risk_ax.spines["bottom"].set_linewidth(0.8)
    risk_ax.spines["right"].set_visible(False)
    risk_ax.spines["top"].set_visible(False)
    risk_ax.grid(False)
    risk_ax.tick_params(axis="x", labelsize=8, colors="black", length=3, width=0.8)
    risk_ax.tick_params(axis="y", length=0, pad=8)
    risk_ax.set_xlabel(x_label, fontsize=10)
    risk_ax.text(x_left, n_rows + 0.85, "Number at risk", ha="left", va="center", fontsize=9, fontweight="bold", color="black")

    for row_idx, group in enumerate(plotted_groups):
        y_pos = n_rows - row_idx - 0.5
        group_df = current_df[current_df[group_col] == group]
        for tick_idx, tick in enumerate(ticks):
            at_risk = int((group_df[time_col] >= tick).sum())
            ha = "left" if tick_idx == 0 else "center"
            risk_ax.text(tick, y_pos, str(at_risk), ha=ha, va="center", fontsize=7, color="black")


def plot_km_with_risk_table(current_df, time_col, event_col, group_col, order, colors, title, x_label, y_label, break_by, output_base):
    present_groups = [g for g in order if g in set(current_df[group_col])]
    valid_groups = []
    for group in present_groups:
        group_df = current_df[current_df[group_col] == group]
        if len(group_df) > 0 and group_df[time_col].notna().all() and group_df[event_col].notna().all():
            valid_groups.append(group)

    if len(valid_groups) < 2:
        print(f"Skip {output_base}: fewer than two valid groups")
        return None

    max_time = current_df[time_col].max()
    ticks, x_upper = get_time_ticks(max_time, break_by)
    zero_pad_fraction = 0.035
    x_left = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * x_upper
    y_top = 1.02
    y_bottom = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * y_top

    fig = plt.figure(figsize=(10.0, 7.8), dpi=300)
    grid = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[4.55, 1.95, 0.86], hspace=0.18)
    ax = fig.add_subplot(grid[0])
    risk_ax = fig.add_subplot(grid[1])
    pair_ax = fig.add_subplot(grid[2])

    plotted_groups = []
    plotted_colors = []
    plotted_labels = []
    label_map = {}
    for i, group in enumerate(order):
        if group not in valid_groups:
            continue
        display_label = f"Group{i + 1}"
        group_df = current_df[current_df[group_col] == group]
        x, y, ci_lower, ci_upper, censor_x, censor_y = km_estimate(group_df[time_col], group_df[event_col])
        ax.fill_between(x, ci_lower, ci_upper, step="post", color=colors[i], alpha=0.16, linewidth=0, zorder=1)
        ax.step(x, y, where="post", color=colors[i], linewidth=1.6, label=f"{display_label}: {group}", zorder=2)
        draw_censor_path(ax, censor_x, censor_y, colors[i], x_upper)
        plotted_groups.append(group)
        plotted_colors.append(colors[i])
        plotted_labels.append(display_label)
        label_map[group] = display_label

    all_pairwise_results, significant_pairwise_results = pairwise_logrank_results(
        current_df, time_col, event_col, group_col, plotted_groups, SIGNIFICANCE_THRESHOLD
    )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_ylim(y_bottom, y_top)
    ax.set_xlim(x_left, x_upper)
    ax.set_xticks(ticks)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.grid(False)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", labelsize=9, colors="black")
    ax.tick_params(axis="x", labelbottom=True, length=3, width=0.8)
    ax.legend(title=None, bbox_to_anchor=(1.02, 1.0), loc="upper left", frameon=False, fontsize=6.6)

    color_map = {label: color for label, color in zip(plotted_labels, plotted_colors)}
    draw_pairwise_results(pair_ax, significant_pairwise_results, label_map, color_map, SIGNIFICANCE_THRESHOLD)

    draw_risk_table(risk_ax, current_df, time_col, group_col, plotted_groups, plotted_colors, plotted_labels, ticks, x_label, x_left, x_upper)

    fig.subplots_adjust(left=0.16, right=0.80, top=0.93, bottom=0.07)
    for ext in ["pdf", "svg"]:
        fig.savefig(f"{output_base}.{ext}", bbox_inches="tight")
    plt.close(fig)

    return {
        "pairwise_logrank_all": serialize_pairwise_results(all_pairwise_results, label_map),
        "significant_pairwise_logrank": serialize_pairwise_results(significant_pairwise_results, label_map),
        "n_significant_pairwise": int(len(significant_pairwise_results)),
        "n_samples": int(current_df.shape[0]),
        "n_groups": int(len(plotted_groups)),
    }


def run_analysis(config):
    os.makedirs(config["output_dir"], exist_ok=True)
    epi, cm = read_scores(config["epi_file"], config["cm_file"], config["sample_style"])
    survival = read_survival(config)

    common_samples = epi.index.intersection(cm.index).intersection(survival.index)
    epi = epi.loc[common_samples].copy()
    cm = cm.loc[common_samples].copy()
    survival = survival.loc[common_samples].copy()

    epi_cols = list(epi.columns)
    cm_cols = list(cm.columns)
    print(config["name"])
    print(f"Matched samples after survival filtering: {len(common_samples)}")
    print(f"Epi columns: {epi_cols}")
    print(f"CM columns: {cm_cols}")

    results = []
    total = len(cm_cols) * len(epi_cols)
    processed = 0

    for cm_col in cm_cols:
        for epi_col in epi_cols:
            processed += 1
            print(f"[{processed}/{total}] {cm_col} vs {epi_col}")
            current_df = survival[[config["time_col"], config["event_col"]]].join(cm[[cm_col]], how="inner").join(epi[[epi_col]], how="inner")
            current_df = current_df.dropna(subset=[config["time_col"], config["event_col"], cm_col, epi_col])
            current_df = current_df[current_df[config["time_col"]] > 0].copy()
            if current_df.empty:
                print("  skip empty data")
                continue

            cm_median = current_df[cm_col].median()
            epi_median = current_df[epi_col].median()
            current_df["cm_group"] = np.where(current_df[cm_col] >= cm_median, "High", "Low")
            current_df["epi_group"] = np.where(current_df[epi_col] >= epi_median, "High", "Low")
            current_df["group"] = "Epi" + epi_col.replace("Epi_", "") + "-" + current_df["epi_group"] + "/" + cm_col + "-" + current_df["cm_group"]

            epi_label = "Epi" + epi_col.replace("Epi_", "")
            order = [
                f"{epi_label}-High/{cm_col}-High",
                f"{epi_label}-High/{cm_col}-Low",
                f"{epi_label}-Low/{cm_col}-High",
                f"{epi_label}-Low/{cm_col}-Low",
            ]

            output_base = os.path.join(config["output_dir"], f"{config['file_prefix']}_{clean_filename(cm_col)}_{clean_filename(epi_col)}")
            plot_result = plot_km_with_risk_table(
                current_df=current_df,
                time_col=config["time_col"],
                event_col=config["event_col"],
                group_col="group",
                order=order,
                colors=PALETTE,
                title=f"{config['plot_title']}: {cm_col} vs {epi_col}",
                x_label=config["x_label"],
                y_label=config["y_label"],
                break_by=config["break_by"],
                output_base=output_base,
            )
            if plot_result is not None:
                row = {
                    "cm": cm_col,
                    "epi": epi_col,
                    "cm_median": cm_median,
                    "epi_median": epi_median,
                    **plot_result,
                }
                for group in order:
                    row[f"n_{group}"] = int((current_df["group"] == group).sum())
                results.append(row)

    summary = pd.DataFrame(results)
    summary_path = os.path.join(config["output_dir"], "km_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"Done: {len(results)} plots saved to {config['output_dir']}")
    print(f"Summary saved to {summary_path}")

CONFIG = {
    "name": "TCGA-KIRC OS",
    "cohort": "tcga",
    "epi_file": "nes_diff.csv",
    "cm_file": "nes_diff_cm.csv",
    "survival_file": "TCGA-KIRC.survival.csv",
    "sample_style": "tcga_dot_to_dash",
    "time_col": "OS.time",
    "event_col": "OS",
    "output_dir": "tcga_kirc_os",
    "file_prefix": "TCGA_KIRC_OS",
    "plot_title": "TCGA-KIRC OS Survival",
    "x_label": "Days",
    "y_label": "OS",
    "break_by": 1000,
}
run_analysis(CONFIG)


## 5. JAVELIN PFS KM analysis using the shared KM functions


In [ ]:
CONFIG = {
    "name": "JAVELIN PFS",
    "cohort": "javelin",
    "epi_file": "nes_diff_javelin.csv",
    "cm_file": "nes_diff_javelin_cm.csv",
    "survival_file": "javelin_clinical_adata.csv",
    "sample_style": "asis",
    "time_col": "PFS_P",
    "raw_event_col": "PFS_P_CNSR",
    "event_col": "PFS_P_event",
    "output_dir": "javelin_pfs",
    "file_prefix": "JAVELIN_PFS",
    "plot_title": "JAVELIN PFS Survival",
    "x_label": "Months",
    "y_label": "PFS",
    "break_by": 5,
}
run_analysis(CONFIG)


## 6. JAVELIN treatment-arm comparison within CM-high and CM-low strata


In [ ]:
os.chdir(FIG6_THERAPY_KM_ROOT)
import os
import re
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["text.usetex"] = False
warnings.filterwarnings("ignore", category=UserWarning)

cm = pd.read_csv('nes_diff_javelin_cm.csv', index_col=0)
survival = pd.read_csv('javelin_clinical_adata.csv', index_col=0)
survival = survival[survival['PFS_P'].notnull()]
survival = survival[survival['PFS_P_CNSR'].notnull()]
survival['PFS_P_event'] = (survival['PFS_P_CNSR'] == 0).astype(int)
surv_cm = survival.join(cm, how='inner')
surv_cm

PALETTE = {
    "arm_1": "#1f77b4",
    "arm_2": "#d62728",
}






def split_high_low(df, cm_col):
    cut = df[cm_col].median()
    high = df[df[cm_col] >= cut].copy()
    low = df[df[cm_col] < cut].copy()
    return [
        ("High", high, "high", cut),
        ("Low", low, "low", cut),
    ]




def draw_byarm_risk_table(risk_ax, df, time_col, arm_col, arm_levels, arm_labels, ticks, x_label, x_left, x_upper):
    n_rows = len(arm_levels)
    risk_ax.set_xlim(x_left, x_upper)
    risk_ax.set_xticks(ticks)
    risk_ax.set_ylim(-0.75, n_rows + 1.2)
    risk_ax.set_yticks([n_rows - row_idx - 0.5 for row_idx in range(n_rows)])
    risk_ax.set_yticklabels([f"Group{i + 1}" for i in range(n_rows)], fontsize=7.0)
    colors = [PALETTE["arm_1"], PALETTE["arm_2"]]
    for tick_label, color in zip(risk_ax.get_yticklabels(), colors):
        tick_label.set_color(color)
    risk_ax.spines["left"].set_visible(True)
    risk_ax.spines["bottom"].set_visible(True)
    risk_ax.spines["left"].set_linewidth(0.8)
    risk_ax.spines["bottom"].set_linewidth(0.8)
    risk_ax.spines["right"].set_visible(False)
    risk_ax.spines["top"].set_visible(False)
    risk_ax.grid(False)
    risk_ax.tick_params(axis="x", labelsize=8, colors="black", length=3, width=0.8)
    risk_ax.tick_params(axis="y", length=0, pad=8)
    risk_ax.set_xlabel(x_label, fontsize=10)
    risk_ax.text(x_left, n_rows + 0.85, "Number at risk", ha="left", va="center", fontsize=9, fontweight="bold", color="black")
    for row_idx, arm in enumerate(arm_levels):
        y_pos = n_rows - row_idx - 0.5
        arm_df = df[df[arm_col] == arm]
        for tick_idx, tick in enumerate(ticks):
            at_risk = int((arm_df[time_col] >= tick).sum())
            risk_ax.text(tick, y_pos, str(at_risk), ha="left" if tick_idx == 0 else "center", va="center", fontsize=7, color="black")




def draw_pairwise_panel(pair_axis, p_value):
    pair_axis.axis("off")
    pair_axis.text(0.0, 0.62, "Pairwise log-rank", fontsize=9, fontweight="bold", ha="left", va="center")
    pair_axis.text(0.0, 0.20, f"Group1 vs Group2 log-rank p = {format_pvalue(p_value)}", fontsize=8, ha="left", va="center")


def plot_javelin_km_by_arm(df, time_col, event_col, arm_col, arm_levels, arm_labels, title, y_label, ticks, output_base):
    df = df.dropna(subset=[time_col, event_col, arm_col]).copy()
    df = df[df[time_col] > 0].copy()
    df = df[df[arm_col].isin(arm_levels)].copy()
    present = [arm for arm in arm_levels if arm in set(df[arm_col])]
    if len(present) < 2:
        print(f"Skip {output_base}: fewer than two arms")
        return None

    max_time = float(df[time_col].max())
    ticks, x_upper = get_time_ticks(max_time)
    zero_pad_fraction = 0.035
    x_left = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * x_upper
    y_top = 1.02
    y_bottom = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * y_top

    fig = plt.figure(figsize=(10.0, 7.8), dpi=300)
    grid = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[4.55, 1.95, 0.86], hspace=0.18)
    ax = fig.add_subplot(grid[0])
    risk_ax = fig.add_subplot(grid[1])
    pair_ax = fig.add_subplot(grid[2])

    kmf = KaplanMeierFitter()
    colors = [PALETTE["arm_1"], PALETTE["arm_2"]]
    for i, arm in enumerate(arm_levels):
        arm_df = df[df[arm_col] == arm]
        if arm_df.empty:
            continue
        kmf.fit(arm_df[time_col], event_observed=arm_df[event_col], label=f"Group{i + 1}: {arm_labels[arm]}")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=colors[i], linewidth=1.6, show_censors=True, censor_styles={"ms": 4, "marker": "+"})

    g1 = df[df[arm_col] == arm_levels[0]]
    g2 = df[df[arm_col] == arm_levels[1]]
    if len(g1) >= 2 and len(g2) >= 2:
        lr = logrank_test(g1[time_col], g2[time_col], event_observed_A=g1[event_col], event_observed_B=g2[event_col])
        p_value = float(lr.p_value)
    else:
        p_value = np.nan

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylim(y_bottom, y_top)
    ax.set_xlim(x_left, x_upper)
    ax.set_xticks(ticks)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.grid(False)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", labelsize=9, colors="black")
    ax.tick_params(axis="x", labelbottom=True, length=3, width=0.8)
    ax.legend(title=None, bbox_to_anchor=(1.02, 1.0), loc="upper left", frameon=False, fontsize=7)

    draw_byarm_risk_table(risk_ax, df, time_col, arm_col, arm_levels, arm_labels, ticks, "Months", x_left, x_upper)
    draw_pairwise_panel(pair_ax, p_value)

    fig.subplots_adjust(left=0.16, right=0.80, top=0.93, bottom=0.07)
    output_base.parent.mkdir(parents=True, exist_ok=True)
    for ext in ["pdf", "svg"]:
        fig.savefig(f"{output_base}.{ext}", bbox_inches="tight")
    plt.close(fig)

    row = {
        "logrank_p": p_value,
        "n_total_after_filter": int(df.shape[0]),
        "pdf": str(output_base.with_suffix(".pdf")),
        "svg": str(output_base.with_suffix(".svg")),
    }
    for arm in arm_levels:
        row[f"n_{clean_filename(arm)}"] = int((df[arm_col] == arm).sum())
    return row


def run_javelin_by_arm_highlow(surv_cm, cm_cols, cohort_label, cohort_dir, arm_col, arm_levels, arm_labels, time_col, event_col, y_label, ticks):
    rows = []
    output_root = Path("immune_therapy_byarm_km_publication_level") / cohort_dir
    for cm_col in cm_cols:
        base = surv_cm[[time_col, event_col, arm_col, cm_col]].dropna().copy()
        base = base[base[time_col] > 0].copy()
        for subset_name, subset_df, subset_short, median_cut in split_high_low(base, cm_col):
            output_base = output_root / clean_filename(cm_col) / subset_short / f"{cohort_dir}_{clean_filename(cm_col)}_{subset_short}_byArm_KM"
            title = f"{cohort_label} | {cm_col} {subset_name}\n{arm_labels[arm_levels[0]]} vs {arm_labels[arm_levels[1]]}; median={median_cut:.3g}"
            plot_result = plot_javelin_km_by_arm(subset_df, time_col, event_col, arm_col, arm_levels, arm_labels, title, y_label, ticks, output_base)
            if plot_result is None:
                continue
            rows.append({
                "cohort": cohort_label,
                "cm": cm_col,
                "subset": subset_name,
                "subset_short": subset_short,
                "median": float(median_cut),
                **plot_result,
            })
    summary = pd.DataFrame(rows)
    output_root.mkdir(parents=True, exist_ok=True)
    summary_path = output_root / "KM_summary_byArm_highlow.csv"
    summary.to_csv(summary_path, index=False)
    print("All plots saved under:", output_root)
    print("Summary saved to:", summary_path)
    return summary

cm_cols = list(cm.columns)
summary_df = run_javelin_by_arm_highlow(
    surv_cm=surv_cm,
    cm_cols=cm_cols,
    cohort_label="JAVELIN PFS",
    cohort_dir="javelin_pfs",
    arm_col="TRT01P",
    arm_levels=["Sunitinib", "Avelumab+Axitinib"],
    arm_labels={"Sunitinib": "Sunitinib", "Avelumab+Axitinib": "Avelumab + Axitinib"},
    time_col="PFS_P",
    event_col="PFS_P_event",
    y_label="Progression-free survival probability",
    ticks=np.array([0, 5, 10, 15, 20, 25]),
)
summary_df


## 7. JAVELIN CM-high versus CM-low KM analysis within each treatment arm


In [ ]:
os.chdir(FIG6_THERAPY_KM_ROOT)
import os
import re
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["text.usetex"] = False
warnings.filterwarnings("ignore", category=UserWarning)

epi = pd.read_csv('nes_diff_javelin.csv', index_col=0)
cm = pd.read_csv('nes_diff_javelin_cm.csv', index_col=0)
survival = pd.read_csv('javelin_clinical_adata.csv', index_col=0)
survival = survival[survival['PFS_P'].notnull()]
survival = survival[survival['PFS_P_CNSR'].notnull()]
survival['PFS_P_event'] = (survival['PFS_P_CNSR'] == 0).astype(int)
survival

surv_cm = survival.join(cm,how='inner')
surv_cm

surv_suni = surv_cm[surv_cm["TRT01P"] == "Sunitinib"].copy()
surv_suni

surv_aveaxi = surv_cm[surv_cm["TRT01P"] == "Avelumab+Axitinib"].copy()
surv_aveaxi

OUT_ROOT = Path("immune_therapy_km_publication_level")
PALETTE = {"High": "#d62728", "Low": "#1f77b4"}






def median_split(df, col):
    cut = df[col].median()
    out = df.copy()
    out["group"] = np.where(out[col] >= cut, "High", "Low")
    return out, {"median": float(cut)}


def quartile_split(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    out = df[(df[col] <= q1) | (df[col] >= q3)].copy()
    out["group"] = np.where(out[col] >= q3, "High", "Low")
    return out, {"q1": float(q1), "q3": float(q3)}


def get_therapy_time_ticks(max_time, break_by=None):
    if not np.isfinite(max_time) or max_time <= 0:
        return np.array([0.0]), 0.0
    steps = np.array([5.0, 10.0, 15.0, 20.0, 25.0, 30.0])
    candidates = []
    max_time = float(max_time)
    for step in steps:
        lower_tick = np.floor(max_time / step) * step
        if lower_tick <= 0:
            continue
        n_nonzero = int(round(lower_tick / step))
        gap = max_time - lower_tick
        count_penalty = 0 if 4 <= n_nonzero <= 7 else abs(n_nonzero - 5)
        candidates.append((gap, count_penalty, abs(n_nonzero - 5), -step, step))
    if not candidates:
        return np.array([0.0]), max_time
    candidates.sort(key=lambda x: (x[0], x[1], x[2], x[3]))
    step = candidates[0][4]
    lower_tick = np.floor(max_time / step) * step
    upper_tick = np.ceil(max_time / step) * step
    lower_gap = max_time - lower_tick
    upper_gap = upper_tick - max_time
    if upper_tick > max_time and upper_gap < lower_gap:
        x_upper = float(upper_tick)
        last_tick = upper_tick
    else:
        x_upper = max_time
        last_tick = lower_tick
    ticks = np.arange(0, last_tick + step * 0.5, step)
    return ticks, x_upper
def draw_therapy_risk_table(risk_ax, df, time_col, groups, ticks, x_label, x_left, x_upper):
    risk_ax.set_xlim(x_left, x_upper)
    risk_ax.set_xticks(ticks)
    risk_ax.set_ylim(-0.75, len(groups) + 1.2)
    risk_ax.set_yticks([len(groups) - row_idx - 0.5 for row_idx in range(len(groups))])
    risk_ax.set_yticklabels([f"Group{i + 1}" for i in range(len(groups))], fontsize=7.0)
    for tick_label, group in zip(risk_ax.get_yticklabels(), groups):
        tick_label.set_color(PALETTE[group])
    risk_ax.spines["left"].set_visible(True)
    risk_ax.spines["bottom"].set_visible(True)
    risk_ax.spines["left"].set_linewidth(0.8)
    risk_ax.spines["bottom"].set_linewidth(0.8)
    risk_ax.spines["right"].set_visible(False)
    risk_ax.spines["top"].set_visible(False)
    risk_ax.grid(False)
    risk_ax.tick_params(axis="x", labelsize=8, colors="black", length=3, width=0.8)
    risk_ax.tick_params(axis="y", length=0, pad=8)
    risk_ax.set_xlabel(x_label, fontsize=10)
    risk_ax.text(x_left, len(groups) + 0.85, "Number at risk", ha="left", va="center", fontsize=9, fontweight="bold", color="black")
    for row_idx, group in enumerate(groups):
        y_pos = len(groups) - row_idx - 0.5
        group_df = df[df["group"] == group]
        for tick_idx, tick in enumerate(ticks):
            at_risk = int((group_df[time_col] >= tick).sum())
            risk_ax.text(tick, y_pos, str(at_risk), ha="left" if tick_idx == 0 else "center", va="center", fontsize=7, color="black")




def plot_javelin_therapy_km(df, time_col, event_col, title, y_label, break_by, output_base):
    groups = [g for g in ["High", "Low"] if g in set(df["group"])]
    if len(groups) < 2:
        print(f"Skip {output_base}: fewer than two groups")
        return None
    max_time = df[time_col].max()
    ticks, x_upper = get_therapy_time_ticks(max_time, break_by)
    zero_pad_fraction = 0.035
    x_left = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * x_upper
    y_top = 1.02
    y_bottom = -(zero_pad_fraction / (1.0 - zero_pad_fraction)) * y_top
    fig = plt.figure(figsize=(10.0, 7.8), dpi=300)
    grid = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[4.55, 1.95, 0.86], hspace=0.18)
    ax = fig.add_subplot(grid[0])
    risk_ax = fig.add_subplot(grid[1])
    pair_ax = fig.add_subplot(grid[2])
    kmf = KaplanMeierFitter()
    for i, group in enumerate(groups):
        group_df = df[df["group"] == group]
        kmf.fit(group_df[time_col], event_observed=group_df[event_col], label=f"Group{i + 1}: {group}")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=PALETTE[group], linewidth=1.6, show_censors=True, censor_styles={"ms": 4, "marker": "+"})
    high = df[df["group"] == "High"]
    low = df[df["group"] == "Low"]
    if len(high) > 0 and len(low) > 0:
        lr = logrank_test(high[time_col], low[time_col], event_observed_A=high[event_col], event_observed_B=low[event_col])
        p_value = float(lr.p_value)
    else:
        p_value = np.nan
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylim(y_bottom, y_top)
    ax.set_xlim(x_left, x_upper)
    ax.set_xticks(ticks)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.grid(False)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", labelsize=9, colors="black")
    ax.tick_params(axis="x", labelbottom=True, length=3, width=0.8)
    ax.legend(title=None, bbox_to_anchor=(1.02, 1.0), loc="upper left", frameon=False, fontsize=7)
    draw_therapy_risk_table(risk_ax, df, time_col, groups, ticks, "Months", x_left, x_upper)
    draw_pairwise_panel(pair_ax, p_value)
    fig.subplots_adjust(left=0.16, right=0.80, top=0.93, bottom=0.07)
    output_base.parent.mkdir(parents=True, exist_ok=True)
    for ext in ["pdf", "svg"]:
        fig.savefig(f"{output_base}.{ext}", bbox_inches="tight")
    plt.close(fig)
    return {
        "logrank_p": p_value,
        "n_total_after_filter": int(df.shape[0]),
        "n_high": int((df["group"] == "High").sum()),
        "n_low": int((df["group"] == "Low").sum()),
        "pdf": str(output_base.with_suffix(".pdf")),
        "svg": str(output_base.with_suffix(".svg")),
    }


def run_javelin_therapy_arm(cohort_dir, cohort_label, arm_label, arm_short, survival_arm, time_col, event_col, cm_df, cm_cols, y_label, break_by):
    rows = []
    common = survival_arm.index.intersection(cm_df.index)
    survival_arm = survival_arm.loc[common].copy()
    cm_arm = cm_df.loc[common].copy()
    for cm_col in cm_cols:
        base = survival_arm[[time_col, event_col]].join(cm_arm[[cm_col]], how="inner").dropna()
        base = base[base[time_col] > 0].copy()
        for split_name, split_func in [("median", median_split), ("quartile", quartile_split)]:
            split_df, cut_info = split_func(base, cm_col)
            out_dir = OUT_ROOT / cohort_dir / arm_short / split_name
            output_base = out_dir / f"{cohort_dir}_{arm_short}_{clean_filename(cm_col)}_{split_name}_KM"
            cut_str = ", ".join(f"{k}={v:.3g}" for k, v in cut_info.items())
            title = f"{cohort_label} ({arm_label}) | {cm_col}\n{split_name} split ({cut_str})"
            plot_result = plot_javelin_therapy_km(split_df, time_col, event_col, title, y_label, break_by, output_base)
            if plot_result is None:
                continue
            rows.append({
                "cohort": cohort_label,
                "arm": arm_label,
                "arm_short": arm_short,
                "cm": cm_col,
                "split": split_name,
                **cut_info,
                **plot_result,
            })
    return rows

cm_cols = list(cm.columns)
arms = [
    ("Avelumab+Axitinib", surv_aveaxi, "avelumab_axitinib", 25),
    ("Sunitinib", surv_suni, "sunitinib", 20),
]

all_rows = []
for arm_name, survival_arm, arm_short, max_tick in arms:
    all_rows.extend(run_javelin_therapy_arm(
        cohort_dir="javelin_pfs",
        cohort_label="JAVELIN PFS",
        arm_label=arm_name,
        arm_short=arm_short,
        survival_arm=survival_arm,
        time_col="PFS_P",
        event_col="PFS_P_event",
        cm_df=cm,
        cm_cols=cm_cols,
        y_label="Progression-free survival probability",
        break_by=max_tick,
    ))
summary_df = pd.DataFrame(all_rows)
summary_path = OUT_ROOT / "javelin_pfs" / "km_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print("All plots saved under:", OUT_ROOT / "javelin_pfs")
print("Summary saved to:", summary_path)
summary_df


## 8. Continuous adjusted clinical models

Rebuild fixed balance scores, the A–C interaction tests and clinical matching from the original inputs, then fit and draw the two final panels.


In [ ]:
# Sources: run_continuum_clinical_models.py; 01_build_axis_scores.py
def prepare_fig6_continuous_inputs(write_outputs=True):
    import math
    import numpy as np
    import pandas as pd
    from scipy import stats
    from statsmodels.duration.hazard_regression import PHReg

    TCGA_EXPRESSION = FIG6_SSGSEA_ROOT / 'gene_symbol_fpkm_star_kirc.csv'
    JAVELIN_EXPRESSION = FIG6_SSGSEA_ROOT / 'javelin_gene_expression_log2tpm.csv'
    TCGA_CLINICAL = FIG6_COX_ROOT / 'TCGA-KIRC.survival.csv'
    JAVELIN_CLINICAL = FIG6_COX_ROOT / 'javelin_clinical_adata.csv'
    OUTDIR = FIG6_REVISION_ROOT / 'results/continuum_clinical_models'
    METHOD_LABELS = {
        "CORE": "Core · treatment × CM1",
        "A": "A · legacy ssGSEA JUN−VIM",
        "B": "B · JUN non-IEG−VIM intermediate",
        "C": "C · JUN non-IEG−VIM paired-JUN",
    }

    def canonical_sample(value: object) -> str:
        if pd.isna(value):
            return ""
        return str(value).strip().replace(".", "-").upper()

    def zscore(values: pd.Series | np.ndarray) -> pd.Series:
        series = pd.Series(values, copy=True, dtype=float)
        mean = series.mean(skipna=True)
        sd = series.std(skipna=True, ddof=0)
        if not np.isfinite(sd) or sd <= 0:
            return pd.Series(np.nan, index=series.index, dtype=float)
        return (series - mean) / sd

    def bh_fdr(values: pd.Series) -> np.ndarray:
        p = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
        q = np.full(len(p), np.nan, dtype=float)
        valid = np.isfinite(p)
        if not valid.any():
            return q
        valid_idx = np.flatnonzero(valid)
        order = valid_idx[np.argsort(p[valid])]
        ranked = p[order] * len(order) / np.arange(1, len(order) + 1)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        q[order] = np.minimum(ranked, 1.0)
        return q

    def load_selected_expression(
        path: Path,
        requested_genes: list[str],
        cohort: str,
        transform: str,
        aliases: dict[str, list[str]],
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Read only requested gene rows while streaming through a gene-by-sample CSV."""
        lookup_symbols = {
            symbol
            for gene in requested_genes
            for symbol in aliases.get(gene, [gene])
        }
        selected: list[pd.DataFrame] = []
        for chunk in pd.read_csv(path, index_col=0, chunksize=4000):
            chunk.index = chunk.index.astype(str)
            keep = chunk.index.isin(lookup_symbols)
            if keep.any():
                selected.append(chunk.loc[keep].copy())
        if not selected:
            raise RuntimeError(f"No requested signature genes found in {path}")

        raw = pd.concat(selected, axis=0)
        raw = raw.apply(pd.to_numeric, errors="coerce")
        # Match the legacy JAVELIN preprocessing for duplicate gene symbols.
        raw = raw.groupby(level=0, sort=False).sum(min_count=1)

        canonical_rows: list[pd.Series] = []
        coverage_rows: list[dict[str, object]] = []
        for gene in requested_genes:
            candidates = aliases.get(gene, [gene])
            matched = next((candidate for candidate in candidates if candidate in raw.index), None)
            if matched is None:
                coverage_rows.append(
                    {
                        "cohort": cohort,
                        "program_gene": gene,
                        "expression_symbol": "",
                        "available": False,
                        "nonzero_variance": False,
                        "transform": transform,
                    }
                )
                continue
            row = raw.loc[matched].copy()
            row.name = gene
            canonical_rows.append(row)
            coverage_rows.append(
                {
                    "cohort": cohort,
                    "program_gene": gene,
                    "expression_symbol": matched,
                    "available": True,
                    "nonzero_variance": bool(pd.to_numeric(row, errors="coerce").std(ddof=0) > 0),
                    "transform": transform,
                }
            )

        expression = pd.DataFrame(canonical_rows)
        if transform == "log2(FPKM+1)":
            expression = np.log2(expression.clip(lower=0) + 1.0)
        elif transform != "stored_log2TPM":
            raise ValueError(f"Unknown expression transform: {transform}")
        return expression, pd.DataFrame(coverage_rows)

    def build_fixed_gene_balance(
        expression: pd.DataFrame,
        jun_genes: list[str],
        vim_genes: list[str],
        cohort: str,
        method: str,
    ) -> pd.DataFrame:
        values = expression.apply(pd.to_numeric, errors="coerce")
        means = values.mean(axis=1)
        sds = values.std(axis=1, ddof=0)
        valid = sds[np.isfinite(sds) & (sds > 0)].index
        gene_z = values.loc[valid].sub(means.loc[valid], axis=0).div(sds.loc[valid], axis=0)

        jun_used = [gene for gene in jun_genes if gene in gene_z.index]
        vim_used = [gene for gene in vim_genes if gene in gene_z.index]
        if len(jun_used) < 3 or len(vim_used) < 3:
            raise RuntimeError(
                f"Insufficient fixed genes for {cohort}: JUN={jun_used}, VIM={vim_used}"
            )

        sample_by_gene = gene_z.T
        out = pd.DataFrame(index=sample_by_gene.index)
        out[f"{method}_JUN_program"] = sample_by_gene[jun_used].mean(axis=1)
        out[f"{method}_VIM_program"] = sample_by_gene[vim_used].mean(axis=1)
        out[f"balance_{method}_raw"] = (
            out[f"{method}_JUN_program"] - out[f"{method}_VIM_program"]
        )
        out[f"balance_{method}_z"] = zscore(out[f"balance_{method}_raw"]).to_numpy()
        out["sample_id_std"] = [canonical_sample(x) for x in out.index]
        out["cohort"] = cohort
        return out.reset_index(drop=True)

    def prepare_axis_scores(
        core: pd.DataFrame,
        fixed_balance: pd.DataFrame,
        cohort: str,
    ) -> pd.DataFrame:
        needed = [
            "sample_id",
            "sample_id_std",
            "z_Epi_JUN",
            "z_Epi_VIM",
            "z_sharedCM_01",
        ]
        sub = core.loc[core["cohort"].eq(cohort), needed].copy()
        for column in ["z_Epi_JUN", "z_Epi_VIM", "z_sharedCM_01"]:
            sub[column] = pd.to_numeric(sub[column], errors="coerce")
        sub["sample_id_std"] = sub["sample_id_std"].map(canonical_sample)
        sub["balance_A_raw"] = sub["z_Epi_JUN"] - sub["z_Epi_VIM"]
        sub["balance_A_z"] = zscore(sub["balance_A_raw"]).to_numpy()
        sub = sub.rename(columns={"z_sharedCM_01": "cm1_z"})

        fixed = fixed_balance.drop(columns=["cohort"], errors="ignore").copy()
        fixed["sample_id_std"] = fixed["sample_id_std"].map(canonical_sample)
        out = sub.merge(fixed, on="sample_id_std", how="inner", validate="one_to_one")
        out.insert(0, "cohort", cohort)
        return out

    def attach_tcga_clinical(axis: pd.DataFrame) -> pd.DataFrame:
        clinical = pd.read_csv(TCGA_CLINICAL)
        clinical["sample_id_std"] = clinical["sample"].map(canonical_sample)
        clinical["time"] = pd.to_numeric(clinical["OS.time"], errors="coerce")
        clinical["event"] = pd.to_numeric(clinical["OS"], errors="coerce")
        keep = ["sample_id_std", "time", "event", "_PATIENT"]
        return axis.merge(clinical[keep], on="sample_id_std", how="inner", validate="one_to_one")

    def attach_javelin_clinical(axis: pd.DataFrame) -> pd.DataFrame:
        clinical = pd.read_csv(JAVELIN_CLINICAL)
        clinical["sample_id_std"] = clinical["ID"].map(canonical_sample)
        clinical["time"] = pd.to_numeric(clinical["PFS_P"], errors="coerce")
        censor = pd.to_numeric(clinical["PFS_P_CNSR"], errors="coerce")
        clinical["event"] = np.where(censor.isna(), np.nan, censor.eq(0).astype(float))
        clinical["treatment_combo"] = clinical["TRT01P"].map(
            {"Sunitinib": 0.0, "Avelumab+Axitinib": 1.0}
        )
        clinical["AGE"] = pd.to_numeric(clinical["AGE"], errors="coerce")
        clinical["sex_male"] = clinical["SEX"].map({"F": 0.0, "M": 1.0})
        clinical["pdl1_positive"] = clinical["PDL1FL"].map({"N": 0.0, "Y": 1.0})
        keep = [
            "sample_id_std",
            "time",
            "event",
            "PFS_P_CNSR",
            "TRT01P",
            "AGE",
            "SEX",
            "PDL1FL",
            "TCGA_cluster",
            "treatment_combo",
            "sex_male",
            "pdl1_positive",
        ]
        merged = axis.merge(clinical[keep], on="sample_id_std", how="inner", validate="one_to_one")
        # Scale age in the actual molecular analysis population (n=726), not in
        # clinical-only trial participants lacking an expression profile.
        merged["age_z"] = zscore(merged["AGE"]).to_numpy()
        return merged

    def fit_nested_cox(
        data: pd.DataFrame,
        full_columns: list[str],
        reduced_columns: list[str],
    ) -> tuple[object, object, pd.DataFrame, float, float]:
        model_data = data[["time", "event", *full_columns]].apply(
            pd.to_numeric, errors="coerce"
        ).dropna()
        if len(model_data) < 30 or model_data["event"].sum() < 10:
            raise RuntimeError("Insufficient complete cases/events for Cox model")
        if any(model_data[column].nunique() < 2 for column in full_columns):
            constants = [column for column in full_columns if model_data[column].nunique() < 2]
            raise RuntimeError(f"Constant Cox predictors: {constants}")

        full_model = PHReg(
            model_data["time"].to_numpy(),
            model_data[full_columns].to_numpy(),
            status=model_data["event"].to_numpy(),
            ties="efron",
        )
        full_fit = full_model.fit(disp=False)
        reduced_model = PHReg(
            model_data["time"].to_numpy(),
            model_data[reduced_columns].to_numpy(),
            status=model_data["event"].to_numpy(),
            ties="efron",
        )
        reduced_fit = reduced_model.fit(disp=False)
        lr_stat = max(0.0, 2.0 * (float(full_fit.llf) - float(reduced_fit.llf)))
        lr_p = float(stats.chi2.sf(lr_stat, df=1))
        return full_fit, reduced_fit, model_data, lr_stat, lr_p

    def coefficient_table(
        fit: object,
        terms: list[str],
        cohort: str,
        method: str,
        endpoint: str,
        n: int,
        events: int,
    ) -> pd.DataFrame:
        params = np.asarray(fit.params, dtype=float)
        ses = np.asarray(fit.bse, dtype=float)
        pvalues = np.asarray(fit.pvalues, dtype=float)
        rows: list[dict[str, object]] = []
        for idx, term in enumerate(terms):
            beta = params[idx]
            se = ses[idx]
            rows.append(
                {
                    "cohort": cohort,
                    "endpoint": endpoint,
                    "balance_method": method,
                    "balance_label": METHOD_LABELS[method],
                    "term": term,
                    "coef": beta,
                    "se": se,
                    "HR": math.exp(beta),
                    "ci_low": math.exp(beta - 1.96 * se),
                    "ci_high": math.exp(beta + 1.96 * se),
                    "wald_p": pvalues[idx],
                    "n": n,
                    "events": events,
                }
            )
        return pd.DataFrame(rows)

    def fit_prespecified_models(
        tcga: pd.DataFrame,
        javelin: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame, dict[tuple[str, str], dict[str, object]]]:
        all_terms: list[pd.DataFrame] = []
        primary_rows: list[dict[str, object]] = []
        fitted: dict[tuple[str, str], dict[str, object]] = {}

        for method in ["A", "B", "C"]:
            balance_col = f"balance_{method}_z"

            tcga_model = tcga.copy()
            tcga_model["balance_z"] = tcga_model[balance_col]
            tcga_model["cm1_x_balance"] = tcga_model["cm1_z"] * tcga_model["balance_z"]
            tcga_terms = ["cm1_z", "balance_z", "cm1_x_balance"]
            tcga_fit, tcga_reduced, tcga_complete, tcga_lr, tcga_lr_p = fit_nested_cox(
                tcga_model, tcga_terms, ["cm1_z", "balance_z"]
            )
            tcga_coef = coefficient_table(
                tcga_fit,
                tcga_terms,
                cohort="TCGA_KIRC",
                method=method,
                endpoint="OS",
                n=len(tcga_complete),
                events=int(tcga_complete["event"].sum()),
            )
            all_terms.append(tcga_coef)
            target = tcga_coef.loc[tcga_coef["term"].eq("cm1_x_balance")].iloc[0]
            primary_rows.append(
                {
                    **target.to_dict(),
                    "test": "CM1 × epithelial balance",
                    "lr_stat_1df": tcga_lr,
                    "lr_p_1df": tcga_lr_p,
                    "full_log_likelihood": float(tcga_fit.llf),
                    "reduced_log_likelihood": float(tcga_reduced.llf),
                }
            )
            fitted[("TCGA_KIRC", method)] = {
                "fit": tcga_fit,
                "terms": tcga_terms,
                "complete": tcga_complete,
            }

            jav_model = javelin.copy()
            jav_model["balance_z"] = jav_model[balance_col]
            jav_model["treatment_x_cm1"] = jav_model["treatment_combo"] * jav_model["cm1_z"]
            jav_model["treatment_x_balance"] = (
                jav_model["treatment_combo"] * jav_model["balance_z"]
            )
            jav_model["cm1_x_balance"] = jav_model["cm1_z"] * jav_model["balance_z"]
            jav_model["treatment_x_cm1_x_balance"] = (
                jav_model["treatment_combo"] * jav_model["cm1_z"] * jav_model["balance_z"]
            )
            jav_terms = [
                "treatment_combo",
                "cm1_z",
                "balance_z",
                "treatment_x_cm1",
                "treatment_x_balance",
                "cm1_x_balance",
                "treatment_x_cm1_x_balance",
                "age_z",
                "sex_male",
                "pdl1_positive",
            ]
            jav_reduced_terms = [term for term in jav_terms if term != "treatment_x_cm1_x_balance"]
            jav_fit, jav_reduced, jav_complete, jav_lr, jav_lr_p = fit_nested_cox(
                jav_model, jav_terms, jav_reduced_terms
            )
            jav_coef = coefficient_table(
                jav_fit,
                jav_terms,
                cohort="JAVELIN",
                method=method,
                endpoint="PFS",
                n=len(jav_complete),
                events=int(jav_complete["event"].sum()),
            )
            all_terms.append(jav_coef)
            target = jav_coef.loc[
                jav_coef["term"].eq("treatment_x_cm1_x_balance")
            ].iloc[0]
            primary_rows.append(
                {
                    **target.to_dict(),
                    "test": "Treatment × CM1 × epithelial balance",
                    "lr_stat_1df": jav_lr,
                    "lr_p_1df": jav_lr_p,
                    "full_log_likelihood": float(jav_fit.llf),
                    "reduced_log_likelihood": float(jav_reduced.llf),
                }
            )
            fitted[("JAVELIN", method)] = {
                "fit": jav_fit,
                "terms": jav_terms,
                "complete": jav_complete,
            }

        terms = pd.concat(all_terms, ignore_index=True)
        primary = pd.DataFrame(primary_rows)
        primary["wald_q_BH_within_cohort"] = primary.groupby("cohort")["wald_p"].transform(
            lambda x: bh_fdr(x)
        )
        primary["lr_q_BH_within_cohort"] = primary.groupby("cohort")["lr_p_1df"].transform(
            lambda x: bh_fdr(x)
        )
        return terms, primary, fitted

    def read_table(path: Path, **kwargs) -> pd.DataFrame:
        path = Path(path)
        if path.name.endswith(".tsv"):
            return pd.read_csv(path, sep="\t", **kwargs)
        return pd.read_csv(path, **kwargs)

    def load_pair(cohort, epi_path, cm_path, sample_col="Name"):
        epi = read_table(epi_path)
        cm = read_table(cm_path)
        epi = epi.rename(columns={sample_col: "sample_id", "sample": "sample_id", "ID": "sample_id"})
        cm = cm.rename(columns={sample_col: "sample_id", "sample": "sample_id", "ID": "sample_id"})
        df = epi.merge(cm, on="sample_id", how="inner")
        df.insert(0, "cohort", cohort)
        return df

    def add_axes(df):
        need = ["Epi_JUN", "Epi_VIM", "Epi_CA9", "joint_01_sharedCM", "joint_10_tCM"]
        for c in need:
            if c not in df.columns:
                df[c] = pd.NA
        df = df.rename(columns={"joint_01_sharedCM": "sharedCM_01", "joint_10_tCM": "tCM_10"})
        for c in ["Epi_JUN", "Epi_VIM", "Epi_CA9", "sharedCM_01", "tCM_10"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df["z_" + c] = df.groupby("cohort")[c].transform(zscore)
        df["JUN_CM1_axis"] = df["z_Epi_JUN"] + df["z_sharedCM_01"]
        df["VIM_CM1_separation_axis"] = df["z_Epi_VIM"] - df["z_sharedCM_01"]
        df["VIM_tCM10_axis"] = df["z_Epi_VIM"] + df["z_tCM_10"]
        df["CA9_tCM10_axis"] = df["z_Epi_CA9"] + df["z_tCM_10"]
        return df

    jun_genes = ['PDK4', 'NLGN1', 'P3H2', 'ZBTB20', 'RORA', 'SOD2', 'VMP1', 'KLF6', 'IRF1']
    vim_intermediate_genes = ['S100A4', 'KLF2', 'KRT7', 'ID1', 'BGN', 'EDN2']
    vim_paired_genes = ['VIM', 'LGALS1', 'LGALS3', 'NUPR1', 'S100A6', 'S100A4']
    requested_genes = list(
        dict.fromkeys([*jun_genes, *vim_intermediate_genes, *vim_paired_genes])
    )
    aliases = {gene: [gene] for gene in requested_genes}
    # JAVELIN uses the historical symbol LEPREL1 for P3H2.
    aliases["P3H2"] = ["P3H2", "LEPREL1"]

    tcga_expression, tcga_coverage = load_selected_expression(
        TCGA_EXPRESSION,
        requested_genes,
        cohort="TCGA_KIRC",
        transform="log2(FPKM+1)",
        aliases=aliases,
    )
    javelin_expression, javelin_coverage = load_selected_expression(
        JAVELIN_EXPRESSION,
        requested_genes,
        cohort="JAVELIN",
        transform="stored_log2TPM",
        aliases=aliases,
    )
    coverage = pd.concat([tcga_coverage, javelin_coverage], ignore_index=True)
    memberships: dict[str, list[str]] = {gene: [] for gene in requested_genes}
    for gene in jun_genes:
        memberships[gene].append("JUN non-IEG")
    for gene in vim_intermediate_genes:
        memberships[gene].append("VIM intermediate")
    for gene in vim_paired_genes:
        memberships[gene].append("VIM paired-JUN")
    coverage["program_membership"] = coverage["program_gene"].map(
        lambda gene: "; ".join(memberships[gene])
    )

    tcga_b = build_fixed_gene_balance(
        tcga_expression,
        jun_genes,
        vim_intermediate_genes,
        "TCGA_KIRC",
        "B",
    )
    tcga_c = build_fixed_gene_balance(
        tcga_expression,
        jun_genes,
        vim_paired_genes,
        "TCGA_KIRC",
        "C",
    )
    javelin_b = build_fixed_gene_balance(
        javelin_expression,
        jun_genes,
        vim_intermediate_genes,
        "JAVELIN",
        "B",
    )
    javelin_c = build_fixed_gene_balance(
        javelin_expression,
        jun_genes,
        vim_paired_genes,
        "JAVELIN",
        "C",
    )

    def merge_fixed(b_frame: pd.DataFrame, c_frame: pd.DataFrame) -> pd.DataFrame:
        return b_frame.merge(
            c_frame.drop(columns=["cohort"], errors="ignore"),
            on="sample_id_std",
            how="inner",
            validate="one_to_one",
        )

    tcga_fixed = merge_fixed(tcga_b, tcga_c)
    javelin_fixed = merge_fixed(javelin_b, javelin_c)


    # Rebuild only the two bulk-cohort profiles needed here, using the same source tables,
    # inner joins, cohort-level standardization, and row order as 01_build_axis_scores.py.
    core = add_axes(pd.concat([
        load_pair('TCGA_KIRC', FIG6_COX_ROOT / 'nes_diff.csv', FIG6_COX_ROOT / 'nes_diff_cm.csv'),
        load_pair('JAVELIN', FIG6_COX_ROOT / 'nes_diff_javelin.csv', FIG6_COX_ROOT / 'nes_diff_javelin_cm.csv'),
    ], ignore_index=True, sort=False))
    core['sample_id_std'] = core['sample_id'].map(canonical_sample)
    tcga_axis = prepare_axis_scores(core, tcga_fixed, 'TCGA_KIRC')
    javelin_axis = prepare_axis_scores(core, javelin_fixed, 'JAVELIN')
    tcga = attach_tcga_clinical(tcga_axis)
    javelin = attach_javelin_clinical(javelin_axis)
    model_terms, primary, _ = fit_prespecified_models(tcga, javelin)
    score_columns = [
        "cohort",
        "sample_id",
        "sample_id_std",
        "cm1_z",
        "z_Epi_JUN",
        "z_Epi_VIM",
        "balance_A_raw",
        "balance_A_z",
        "B_JUN_program",
        "B_VIM_program",
        "balance_B_raw",
        "balance_B_z",
        "C_JUN_program",
        "C_VIM_program",
        "balance_C_raw",
        "balance_C_z",
        "time",
        "event",
    ]
    tcga_scores = tcga[score_columns].copy()
    javelin_extra = ["TRT01P", "AGE", "SEX", "PDL1FL", "TCGA_cluster"]
    javelin_scores = javelin[[*score_columns, *javelin_extra]].copy()
    combined_scores = pd.concat([tcga_scores, javelin_scores], ignore_index=True, sort=False)


    if write_outputs:
        OUTDIR.mkdir(parents=True, exist_ok=True)
        combined_scores.to_csv(OUTDIR / 'continuous_balance_scores.csv.gz', index=False)
        coverage.to_csv(OUTDIR / 'fixed_gene_signature_coverage.csv', index=False)
        primary.to_csv(OUTDIR / 'primary_interaction_tests.csv', index=False)
    return combined_scores, primary, model_terms, coverage


fig6_continuous_scores, fig6_primary_tests, fig6_model_terms, fig6_gene_coverage = prepare_fig6_continuous_inputs()


In [ ]:
# Source: run_tcga_adjusted_clinical.py (cached GDC snapshot)
def prepare_fig6_tcga_clinical_input(write_outputs=True):
    import gzip
    import json
    from typing import Any
    import numpy as np
    import pandas as pd

    OUTDIR = FIG6_REVISION_ROOT / 'results/tcga_adjusted_clinical'
    SCORE_INPUT = FIG6_REVISION_ROOT / 'results/continuum_clinical_models/continuous_balance_scores.csv.gz'
    def canonical_sample(value: object) -> str:
        if pd.isna(value):
            return ""
        return str(value).strip().replace(".", "-").upper()

    def zscore(values: pd.Series) -> pd.Series:
        numeric = pd.to_numeric(values, errors="coerce")
        sd = numeric.std(skipna=True, ddof=0)
        if not np.isfinite(sd) or sd <= 0:
            raise RuntimeError("Cannot standardize a constant or missing covariate")
        return (numeric - numeric.mean(skipna=True)) / sd

    def load_scores() -> pd.DataFrame:
        data = pd.read_csv(SCORE_INPUT)
        data = data.loc[data["cohort"].eq("TCGA_KIRC")].copy()
        required = {
            "sample_id_std",
            "time",
            "event",
            "cm1_z",
            "balance_C_z",
        }
        missing = required.difference(data.columns)
        if missing:
            raise RuntimeError(f"Missing score columns: {sorted(missing)}")
        data["sample_id_std"] = data["sample_id_std"].map(canonical_sample)
        data["patient_id"] = data["sample_id_std"].str.slice(0, 12)
        if data["patient_id"].duplicated().any():
            duplicated = data.loc[data["patient_id"].duplicated(False), "patient_id"].tolist()
            raise RuntimeError(f"Multiple molecular samples per patient: {duplicated[:10]}")
        for column in ["time", "event", "cm1_z", "balance_C_z"]:
            data[column] = pd.to_numeric(data[column], errors="coerce")
        if data[list(required - {"sample_id_std"})].isna().any().any():
            raise RuntimeError("Missing OS or fixed score values in TCGA molecular input")
        return data

    def extract_primary_clinical(
        payload: dict[str, Any], patient_ids: set[str]
    ) -> pd.DataFrame:
        hits = payload.get("data", {}).get("hits", [])
        total = payload.get("data", {}).get("pagination", {}).get("total")
        if total is None or len(hits) != int(total):
            raise RuntimeError(
                f"Incomplete GDC pagination: returned {len(hits)} of total {total}"
            )
        by_submitter: dict[str, dict[str, Any]] = {}
        for hit in hits:
            submitter = canonical_sample(hit.get("submitter_id"))
            if submitter in by_submitter:
                raise RuntimeError(f"Duplicate GDC case submitter_id: {submitter}")
            by_submitter[submitter] = hit

        missing_cases = sorted(patient_ids.difference(by_submitter))
        if missing_cases:
            raise RuntimeError(f"Molecular patients absent from GDC: {missing_cases[:10]}")

        rows: list[dict[str, Any]] = []
        for patient_id in sorted(patient_ids):
            hit = by_submitter[patient_id]
            diagnoses = hit.get("diagnoses") or []
            primary = [
                diagnosis
                for diagnosis in diagnoses
                if diagnosis.get("diagnosis_is_primary_disease") is True
            ]
            # Do not guess among diagnoses: every matched patient must have exactly
            # one explicitly marked primary diagnosis.
            if len(primary) != 1:
                raise RuntimeError(
                    f"{patient_id}: expected exactly one primary diagnosis, found {len(primary)}"
                )
            diagnosis = primary[0]
            demographic = hit.get("demographic") or {}
            rows.append(
                {
                    "patient_id": patient_id,
                    "gdc_case_id": hit.get("case_id"),
                    "gdc_primary_diagnosis_id": diagnosis.get("diagnosis_id"),
                    "age_at_index_years": demographic.get("age_at_index"),
                    "sex_at_birth": demographic.get("sex_at_birth"),
                    "age_at_diagnosis_days": diagnosis.get("age_at_diagnosis"),
                    "pathologic_stage": diagnosis.get("ajcc_pathologic_stage"),
                    "tumor_grade": diagnosis.get("tumor_grade"),
                    "ajcc_staging_system_edition": diagnosis.get(
                        "ajcc_staging_system_edition"
                    ),
                    "diagnosis_classification": diagnosis.get(
                        "classification_of_tumor"
                    ),
                    "diagnosis_days_to_diagnosis": diagnosis.get("days_to_diagnosis"),
                }
            )
        return pd.DataFrame(rows)

    def build_analysis_frame(scores: pd.DataFrame, clinical: pd.DataFrame) -> pd.DataFrame:
        data = scores.merge(clinical, on="patient_id", how="left", validate="one_to_one")
        if data["gdc_case_id"].isna().any():
            raise RuntimeError("Unexpected unmatched GDC records after validated merge")

        data["age_at_index_years"] = pd.to_numeric(
            data["age_at_index_years"], errors="coerce"
        )
        data["age_z"] = zscore(data["age_at_index_years"])
        data["sex_male"] = data["sex_at_birth"].map({"female": 0.0, "male": 1.0})
        data["cm1_x_balance_C"] = data["cm1_z"] * data["balance_C_z"]

        allowed_stage = ["Stage I", "Stage II", "Stage III", "Stage IV"]
        stage_valid = data["pathologic_stage"].isin(allowed_stage)
        for stage, roman in [
            ("Stage II", "II"),
            ("Stage III", "III"),
            ("Stage IV", "IV"),
        ]:
            data[f"stage_{roman}"] = (
                data["pathologic_stage"].eq(stage).astype(float).where(stage_valid)
            )
        data["stage_stratum"] = data["pathologic_stage"].where(stage_valid)

        data["grade_ordinal"] = data["tumor_grade"].map(
            {"G1": 1.0, "G2": 2.0, "G3": 3.0, "G4": 4.0}
        )
        return data

    scores = load_scores()
    # Preserve the exact GDC snapshot used for the figure; do not silently refetch changed records.
    snapshot = OUTDIR / 'gdc_tcga_kirc_cases.json.gz'
    with gzip.open(snapshot, 'rt', encoding='utf-8') as handle:
        payload = json.load(handle)
    clinical = extract_primary_clinical(payload, set(scores['patient_id']))
    data = build_analysis_frame(scores, clinical)
    if write_outputs:
        OUTDIR.mkdir(parents=True, exist_ok=True)
        data.to_csv(OUTDIR / 'tcga_clinical_score_match.csv.gz', index=False)
    return data


fig6_tcga_clinical = prepare_fig6_tcga_clinical_input()


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from statsmodels.duration.hazard_regression import PHReg

FIG6_PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
FIG6_MODEL_INPUT = FIG6_PROJECT_ROOT / "review_revision_20260819/results"
FIG6_SOURCE_DIR = FIG6_MODEL_INPUT / "rebuilt_main_figures/Figure6_extensions"
FIG6_OUTPUT_DIR_A = FIG6_REVISION_ROOT / "ppt_lite_revision_20260828/figures"
FIG6_OUTPUT_DIR_B = FIG6_REVISION_ROOT / "standalone_human_visuals_20260823/Figure6"
FIG6_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
FIG6_OUTPUT_DIR_A.mkdir(parents=True, exist_ok=True)
FIG6_OUTPUT_DIR_B.mkdir(parents=True, exist_ok=True)
TCGA_MODEL_INPUT = FIG6_MODEL_INPUT / "tcga_adjusted_clinical/tcga_clinical_score_match.csv.gz"
JAVELIN_MODEL_INPUT = FIG6_MODEL_INPUT / "continuum_clinical_models/continuous_balance_scores.csv.gz"
JAVELIN_PRIMARY_TESTS = FIG6_MODEL_INPUT / "continuum_clinical_models/primary_interaction_tests.csv"
TCGA_TERMS = ["cm1_z","balance_C_z","cm1_x_balance_C","age_z","sex_male","stage_II","stage_III","stage_IV"]
JAVELIN_TERMS = ["treatment_combo","cm1_z","balance_z","treatment_x_cm1","treatment_x_balance","cm1_x_balance","treatment_x_cm1_x_balance","age_z","sex_male","pdl1_positive"]
BALANCE_LEVELS = [-1.0, 0.0, 1.0]

def breslow_cumulative_hazard(time, event, linear_predictor, grid_days):
    event_times=np.unique(time[event==1]); event_times.sort(); risk=np.exp(np.clip(linear_predictor,-35,35)); increments=[]
    for event_time in event_times:
        increments.append(np.count_nonzero((time==event_time)&(event==1))/risk[time>=event_time].sum())
    cumulative=np.cumsum(increments); positions=np.searchsorted(event_times,grid_days,side="right")-1; output=np.zeros(len(grid_days)); valid=positions>=0; output[valid]=cumulative[positions[valid]]; return output

def standardized_survival(time,event,matrix,beta,grid_days):
    baseline=breslow_cumulative_hazard(time,event,matrix@beta,grid_days); curves=[]
    for balance in BALANCE_LEVELS:
        standardized=matrix.copy(); standardized[:,TCGA_TERMS.index("balance_C_z")]=balance; standardized[:,TCGA_TERMS.index("cm1_x_balance_C")]=standardized[:,TCGA_TERMS.index("cm1_z")]*balance
        relative_risk=np.exp(np.clip(standardized@beta,-35,35)); curves.append(np.exp(-baseline[:,None]*relative_risk[None,:]).mean(axis=1))
    return np.asarray(curves)

def fit_tcga_adjusted_curves(n_bootstrap=240):
    raw=pd.read_csv(TCGA_MODEL_INPUT); complete=raw[["time","event",*TCGA_TERMS]].apply(pd.to_numeric,errors="coerce").dropna()
    time=complete.time.to_numpy(float); event=complete.event.to_numpy(float); matrix=complete[TCGA_TERMS].to_numpy(float)
    fit=PHReg(time,matrix,status=event,ties="efron").fit(disp=False); beta=np.asarray(fit.params,float); covariance=np.asarray(fit.cov_params(),float)
    grid_years=np.linspace(0,8,161); grid_days=grid_years*365.25; estimate=standardized_survival(time,event,matrix,beta,grid_days)
    seed_sequence=np.random.SeedSequence(20260820); seeds=[int(seed.generate_state(1)[0]) for seed in seed_sequence.spawn(n_bootstrap)]; bootstrap=[]
    for seed in seeds:
        rng=np.random.default_rng(seed); index=rng.integers(0,len(time),size=len(time)); sampled_time=time[index]; sampled_event=event[index]; sampled_matrix=matrix[index]
        if sampled_event.sum()<30: continue
        try:
            sampled_fit=PHReg(sampled_time,sampled_matrix,status=sampled_event,ties="efron").fit(disp=False); sampled_beta=np.asarray(sampled_fit.params,float)
            if np.isfinite(sampled_beta).all(): bootstrap.append(standardized_survival(sampled_time,sampled_event,sampled_matrix,sampled_beta,grid_days))
        except Exception:
            continue
    if len(bootstrap)<max(50,int(.8*n_bootstrap)): raise RuntimeError(f"Only {len(bootstrap)}/{n_bootstrap} TCGA bootstrap fits succeeded")
    bootstrap=np.stack(bootstrap); low=np.minimum(np.quantile(bootstrap,.025,axis=0),estimate); high=np.maximum(np.quantile(bootstrap,.975,axis=0),estimate)
    labels={-1.0:"VIM-leaning (-1 SD)",0.0:"Central (0 SD)",1.0:"JUN-leaning (+1 SD)"}; rows=[]
    for level_index,balance in enumerate(BALANCE_LEVELS):
        for time_index,years in enumerate(grid_years): rows.append({"time_years":years,"time_days":grid_days[time_index],"balance_C_z":balance,"balance_label":labels[balance],"standardized_OS":estimate[level_index,time_index],"ci_low":low[level_index,time_index],"ci_high":high[level_index,time_index]})
    curves=pd.DataFrame(rows); coefficients=[]
    for index,term in enumerate(TCGA_TERMS):
        se=math.sqrt(covariance[index,index]); coefficients.append({"term":term,"coef":beta[index],"se":se,"HR":math.exp(beta[index]),"ci_low":math.exp(beta[index]-1.96*se),"ci_high":math.exp(beta[index]+1.96*se),"wald_p":float(fit.pvalues[index]),"n":len(complete),"events":int(event.sum())})
    return curves,pd.DataFrame(coefficients)

def prepare_javelin_model_data():
    data=pd.read_csv(JAVELIN_MODEL_INPUT); data=data[data.cohort.eq("JAVELIN")].copy(); data["treatment_combo"]=data.TRT01P.map({"Sunitinib":0.0,"Avelumab+Axitinib":1.0}); data["sex_male"]=data.SEX.map({"F":0.0,"M":1.0}); data["pdl1_positive"]=data.PDL1FL.map({"N":0.0,"Y":1.0})
    age=pd.to_numeric(data.AGE,errors="coerce"); data["age_z"]=(age-age.mean())/age.std(ddof=0); data["balance_z"]=pd.to_numeric(data.balance_C_z,errors="coerce")
    data["treatment_x_cm1"]=data.treatment_combo*data.cm1_z; data["treatment_x_balance"]=data.treatment_combo*data.balance_z; data["cm1_x_balance"]=data.cm1_z*data.balance_z; data["treatment_x_cm1_x_balance"]=data.treatment_combo*data.cm1_z*data.balance_z
    return data

def fit_javelin_treatment_surface():
    data=prepare_javelin_model_data(); complete=data[["time","event",*JAVELIN_TERMS]].apply(pd.to_numeric,errors="coerce").dropna(); fit=PHReg(complete.time.to_numpy(),complete[JAVELIN_TERMS].to_numpy(),status=complete.event.to_numpy(),ties="efron").fit(disp=False)
    beta=np.asarray(fit.params,float); covariance=np.asarray(fit.cov_params(),float); positions={term:index for index,term in enumerate(JAVELIN_TERMS)}
    x=np.linspace(*np.quantile(complete.cm1_z,[.025,.975]),151); y=np.linspace(*np.quantile(complete.balance_z,[.025,.975]),151); xx,yy=np.meshgrid(x,y); contrasts=np.zeros((xx.size,len(JAVELIN_TERMS)))
    contrasts[:,positions["treatment_combo"]]=1; contrasts[:,positions["treatment_x_cm1"]]=xx.ravel(); contrasts[:,positions["treatment_x_balance"]]=yy.ravel(); contrasts[:,positions["treatment_x_cm1_x_balance"]]=(xx*yy).ravel()
    log_hr=contrasts@beta; variance=np.einsum("ij,jk,ik->i",contrasts,covariance,contrasts); se=np.sqrt(np.maximum(variance,0)); surface=pd.DataFrame({"cm1_z":xx.ravel(),"balance_C_z":yy.ravel(),"log_HR":log_hr,"HR":np.exp(log_hr),"ci_low":np.exp(log_hr-1.96*se),"ci_high":np.exp(log_hr+1.96*se)})
    observed_contrasts=np.zeros((len(complete),len(JAVELIN_TERMS))); observed_contrasts[:,positions["treatment_combo"]]=1; observed_contrasts[:,positions["treatment_x_cm1"]]=complete.cm1_z; observed_contrasts[:,positions["treatment_x_balance"]]=complete.balance_z; observed_contrasts[:,positions["treatment_x_cm1_x_balance"]]=complete.cm1_z*complete.balance_z
    observed_log_hr=observed_contrasts@beta; observed_variance=np.einsum("ij,jk,ik->i",observed_contrasts,covariance,observed_contrasts); observed_se=np.sqrt(np.maximum(observed_variance,0)); observed=complete[["cm1_z","balance_z"]].rename(columns={"balance_z":"balance_C_z"}).copy(); observed["modeled_treatment_HR"]=np.exp(observed_log_hr); observed["ci_low"]=np.exp(observed_log_hr-1.96*observed_se); observed["ci_high"]=np.exp(observed_log_hr+1.96*observed_se)
    coefficients=[]
    for index,term in enumerate(JAVELIN_TERMS):
        term_se=math.sqrt(covariance[index,index]); coefficients.append({"term":term,"coef":beta[index],"se":term_se,"HR":math.exp(beta[index]),"ci_low":math.exp(beta[index]-1.96*term_se),"ci_high":math.exp(beta[index]+1.96*term_se),"wald_p":float(fit.pvalues[index]),"n":len(complete),"events":int(complete.event.sum())})
    return surface,observed,pd.DataFrame(coefficients)

tcga_curves,tcga_coefficients=fit_tcga_adjusted_curves(n_bootstrap=240)
javelin_surface,javelin_observed,javelin_coefficients=fit_javelin_treatment_surface()
assert int(tcga_coefficients.iloc[0].n)==522 and int(javelin_coefficients.iloc[0].n)==726
frozen_triple=pd.read_csv(JAVELIN_PRIMARY_TESTS).query("cohort == 'JAVELIN' and balance_method == 'C' and term == 'treatment_x_cm1_x_balance'").iloc[0]; refit_triple=javelin_coefficients.set_index("term").loc["treatment_x_cm1_x_balance"]; assert np.isclose(frozen_triple.coef,refit_triple.coef,atol=1e-8)
tcga_curves.to_csv(FIG6_SOURCE_DIR / "TCGA_adjusted_marginal_OS_source_data.csv",index=False); tcga_coefficients.to_csv(FIG6_SOURCE_DIR / "TCGA_adjusted_model_coefficients.csv",index=False)
javelin_surface.to_csv(FIG6_SOURCE_DIR / "JAVELIN_treatment_benefit_surface_source_data.csv.gz",index=False,compression="gzip"); javelin_observed.to_csv(FIG6_SOURCE_DIR / "JAVELIN_observed_profile_treatment_contrasts.csv",index=False); javelin_coefficients.to_csv(FIG6_SOURCE_DIR / "JAVELIN_hierarchical_model_coefficients.csv",index=False)


In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import FancyArrowPatch
from scipy import stats
import numpy as np
import pandas as pd

FIG6_COLORS = {
    "ca9": "#6B5CA5",
    "vim": "#2A9D8F",
    "jun": "#E67E22",
    "cm1": "#3D6D9C",
    "spp1": "#C94F49",
    "dark": "#263238",
    "mid": "#66717B",
    "light": "#E9EDF0",
    "gray": "#9AA3AA",
    "treatment": "#2F6FB0",
}

def configure_fig6a() -> None:
    regular = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF")
    bold = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF")
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    mpl.rcParams.update(
        {
            "font.family": "Arial" if regular.exists() else "DejaVu Sans",
            "font.size": 8.5,
            "axes.titlesize": 9.5,
            "axes.labelsize": 9.0,
            "xtick.labelsize": 8.0,
            "ytick.labelsize": 8.0,
            "legend.fontsize": 8.0,
            "axes.linewidth": 0.75,
            "xtick.major.width": 0.75,
            "ytick.major.width": 0.75,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
        }
    )

def header_fig6a(fig: plt.Figure, letter: str, title: str, subtitle: str) -> None:
    fig.text(0.035, 0.965, letter, fontsize=16, fontweight="bold", color=FIG6_COLORS["dark"], va="top")
    fig.text(0.095, 0.965, title, fontsize=12.5, fontweight="bold", color=FIG6_COLORS["dark"], va="top")
    fig.text(0.095, 0.910, subtitle, fontsize=8.5, color=FIG6_COLORS["mid"], va="top")

def clean_fig6a(ax: mpl.axes.Axes, grid: str | None = None) -> None:
    ax.spines[["top", "right"]].set_visible(False)
    if grid:
        ax.grid(axis=grid, color="#E5EAED", lw=0.65)
        ax.set_axisbelow(True)

def configure_fig6b() -> None:
    regular = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF")
    bold = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF")
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    mpl.rcParams.update(
        {
            "font.family": "Arial" if regular.exists() else "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9.5,
            "xtick.labelsize": 8.5,
            "ytick.labelsize": 8.5,
            "legend.fontsize": 8.5,
            "axes.linewidth": 0.75,
            "xtick.major.width": 0.75,
            "ytick.major.width": 0.75,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
        }
    )

def header_fig6b(fig: plt.Figure, letter: str, title: str, subtitle: str = "") -> None:
    fig.text(0.035, 0.965, letter, fontsize=17, fontweight="bold", color=FIG6_COLORS["dark"], va="top")
    fig.text(0.10, 0.965, title, fontsize=14, fontweight="bold", color=FIG6_COLORS["dark"], va="top")
    if subtitle:
        fig.text(0.10, 0.915, subtitle, fontsize=9.5, color=FIG6_COLORS["mid"], va="top")

def draw_figure6a():
    configure_fig6a()
    curves = pd.read_csv(FIG6_SOURCE_DIR / 'TCGA_adjusted_marginal_OS_source_data.csv')
    coefficients = pd.read_csv(FIG6_SOURCE_DIR / 'TCGA_adjusted_model_coefficients.csv').set_index('term')
    fig, ax = plt.subplots(figsize=(7.09, 4.65))
    fig.subplots_adjust(left=0.13, right=0.84, bottom=0.17, top=0.80)
    header_fig6a(
        fig,
        "A",
        "JUN-leaning epithelial balance is associated with longer adjusted survival",
        "Model-standardized overall survival at −1 and +1 SD; adjusted for age, sex and stage; n=522",
    )
    display = [(-1.0, "VIM-leaning (−1 SD)", FIG6_COLORS["vim"]), (1.0, "JUN-leaning (+1 SD)", FIG6_COLORS["jun"])]
    five_values = {}
    for level, label, color in display:
        data = curves.loc[curves["balance_C_z"].eq(level)].sort_values("time_years")
        x = data["time_years"].to_numpy(float)
        y = data["standardized_OS"].to_numpy(float)
        ax.fill_between(x, data["ci_low"], data["ci_high"], color=color, alpha=0.14, linewidth=0)
        ax.plot(x, y, color=color, lw=2.7, solid_capstyle="round")
        last = data.iloc[-1]
        ax.text(8.10, last["standardized_OS"], label, color=color, va="center", fontsize=8.8, fontweight="bold")
        five = data.iloc[(data["time_years"] - 5).abs().argmin()]
        five_values[level] = float(five["standardized_OS"])
        ax.scatter(5, five["standardized_OS"], s=42, color=color, edgecolor="white", linewidth=0.8, zorder=4)
        ax.text(4.88, five["standardized_OS"], f"{five['standardized_OS']:.0%}", ha="right", va="center", color=color, fontweight="bold")
    ax.axvline(5, color="#BBC4CA", lw=0.8, ls=(0, (3, 3)), zorder=0)
    low, high = five_values[-1.0], five_values[1.0]
    bracket = FancyArrowPatch((5.22, low), (5.22, high), arrowstyle="<->", mutation_scale=10, color=FIG6_COLORS["dark"], lw=1.0)
    ax.add_patch(bracket)
    ax.text(5.32, (low + high) / 2, f"+{100*(high-low):.0f} pp", va="center", color=FIG6_COLORS["dark"], fontweight="bold")
    coef = coefficients.loc["balance_C_z"]
    ax.text(0.02, 0.045, f"HR {coef['HR']:.2f} per SD toward JUN (95% CI {coef['ci_low']:.2f}–{coef['ci_high']:.2f})", transform=ax.transAxes, color=FIG6_COLORS["dark"], fontweight="bold")
    ax.set_xlim(0, 9.20)
    ax.set_ylim(0.20, 1.02)
    ax.set_xlabel("Time from diagnosis (years)")
    ax.set_ylabel("Adjusted survival probability")
    ax.set_xticks([0, 2, 4, 6, 8])
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    clean_fig6a(ax, grid="both")
    FIG6_OUTPUT_DIR_A.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG6_OUTPUT_DIR_A / "Figure6A_two_adjusted_survival_curves.pdf", dpi=450, bbox_inches="tight")
    plt.close(fig)

def draw_figure6b() -> None:
    configure_fig6b()
    surface = pd.read_csv(FIG6_SOURCE_DIR / "JAVELIN_treatment_benefit_surface_source_data.csv.gz")
    observed = pd.read_csv(FIG6_SOURCE_DIR / "JAVELIN_observed_profile_treatment_contrasts.csv")
    tests = pd.read_csv((FIG6_MODEL_INPUT / "continuum_clinical_models") / "primary_interaction_tests.csv")
    row = tests.loc[
        tests["cohort"].eq("JAVELIN")
        & tests["balance_method"].eq("C")
        & tests["term"].eq("treatment_x_cm1_x_balance")
    ].iloc[0]
    x = np.sort(surface["cm1_z"].unique())
    y = np.sort(surface["balance_C_z"].unique())
    hr = surface.pivot(index="balance_C_z", columns="cm1_z", values="HR").loc[y, x].to_numpy()
    xx, yy = np.meshgrid(x, y)

    fig, ax = plt.subplots(figsize=(7.3, 6.0))
    fig.subplots_adjust(left=0.15, right=0.88, bottom=0.14, top=0.80)
    header_fig6b(
        fig,
        "B",
        "Treatment benefit depends jointly on CM1 and epithelial state",
        f"JAVELIN Renal 101; continuous three-way interaction, likelihood-ratio P={row['lr_p_1df']:.4f}",
    )
    image = ax.pcolormesh(
        xx,
        yy,
        np.log2(hr),
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vmin=-2.0, vcenter=0.0, vmax=2.0),
        shading="auto",
        rasterized=True,
    )
    unity = ax.contour(xx, yy, hr, levels=[1.0], colors="#27343A", linewidths=1.7)
    ax.clabel(unity, inline=True, fmt={1.0: "equal treatment hazard"}, fontsize=7.5)

    points = observed[["cm1_z", "balance_C_z"]].dropna().to_numpy().T
    kde = stats.gaussian_kde(points)
    density = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    observed_density = kde(points)
    levels = np.unique(np.quantile(observed_density, [0.20, 0.50, 0.80]))
    if len(levels) >= 2:
        ax.contour(xx, yy, density, levels=levels, colors="white", linewidths=0.8, alpha=0.78)

    ax.text(0.04, 0.92, "Combination\nfavored", transform=ax.transAxes, ha="left", va="top", color="white", fontsize=9.2, fontweight="bold")
    ax.text(0.96, 0.92, "Sunitinib\nfavored", transform=ax.transAxes, ha="right", va="top", color="#5A2525", fontsize=9.2, fontweight="bold")
    ax.text(0.04, 0.08, "Sunitinib\nfavored", transform=ax.transAxes, ha="left", va="bottom", color="#5A2525", fontsize=9.2, fontweight="bold")
    ax.text(0.96, 0.08, "Combination\nfavored", transform=ax.transAxes, ha="right", va="bottom", color="white", fontsize=9.2, fontweight="bold")

    cbar = fig.colorbar(image, ax=ax, fraction=0.048, pad=0.035)
    cbar.set_ticks([-2, -1, 0, 1, 2])
    cbar.set_ticklabels(["0.25", "0.5", "1", "2", "4"])
    cbar.set_label("PFS hazard ratio\ncombination vs sunitinib")
    cbar.outline.set_linewidth(0.6)
    ax.set_xlabel("CM1 score  →")
    ax.set_ylabel("VIM-leaning  ←  epithelial balance  →  JUN-leaning")
    ax.tick_params(length=3)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)
        spine.set_color("#9AA5AC")
    FIG6_OUTPUT_DIR_B.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG6_OUTPUT_DIR_B / "Figure6B_treatment_benefit_landscape.pdf", bbox_inches="tight")
    plt.close(fig)

draw_figure6a()
draw_figure6b()
